# LoRA Layer-wise Factuality — Figures & Tables

Interactive, cell-by-cell rebuild of `src/lora_lens/visualize.py`. Each figure
or table gets its own cell so you can re-run, tweak, or skip any single output
without regenerating the rest. Plots are displayed inline **and** saved as PDF
under `<output_dir>/figures/...`, matching the layout the pipeline's
`visualize` stage already produces. Tables are rendered as `pandas`
DataFrames instead of LaTeX snippets.

Change `RESULTS_DIR` below to point at a different run; everything else
recomputes from that.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sstats
from IPython.display import display, Image
import io

from lora_lens import visualize as viz
from lora_lens.trajectory import moderator_data, CLASS_ORDER

# viz's module-level plt.rcParams.update(...) already applied the academic
# style (serif font, no top/right spines, light grid, tight 300dpi PDFs) —
# reuse its palette/labels/markers so the notebook stays visually consistent
# with any figures the script itself still produces.
COLORS = viz.COLORS
CONDITION_LABELS = viz.CONDITION_LABELS
LINE_STYLES = viz.LINE_STYLES
MARKERS = viz.MARKERS
COND_ORDER = viz.COND_ORDER
MARKER_EVERY = viz.MARKER_EVERY
LENSES = viz.LENSES


In [ ]:
# --- Run selection -----------------------------------------------------
# Point this at any results/<run> directory that holds layerwise.parquet
# directly (flat layout, same convention as `python -m lora_lens.visualize`).
RESULTS_DIR = Path("../results/run31_07_02")
OUTPUT_DIR = RESULTS_DIR          # conditions.parquet, if present, sits beside it
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results:  {RESULTS_DIR.resolve()}")
print(f"Figures:  {FIGURES_DIR.resolve()}")


In [ ]:
def show_and_save(fig, stem, *, lens=None, prefix="new_version"):
    """Display fig inline, save it as PDF under FIGURES_DIR, then close.

    Renders a PNG preview through an explicit buffer rather than relying on
    IPython's Figure repr formatter, since that formatter is only registered
    by the `inline` matplotlib backend -- and visualize.py's `matplotlib.use
    ("Agg")` (imported for headless PDF generation) would otherwise leave
    plain `display(fig)` calls silently falling back to a text repr.
    """
    out_dir = FIGURES_DIR / lens if lens else FIGURES_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{prefix}_{stem}.pdf"
    fig.savefig(out)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    display(Image(data=buf.getvalue()))
    print(f"[notebook] saved {out}")


## Population reference

Every cell states its population explicitly, but the recurring vocabulary is:

- **variant** — `base` (pre-LoRA), `step_NNN` (intermediate checkpoint), `final`
  (last LoRA checkpoint).
- **condition** — `known` / `latent` (top-5 pre-LoRA) / `unknown`: real
  CounterFact facts stratified by whether the base model already knew them;
  `synthetic`: fabricated pseudo-entity facts the base model cannot have seen.
  `existing` = known ∪ latent ∪ unknown.
- **prompt_type** — `train` (facts LoRA was fine-tuned on) vs `paraphrase`
  (held-out generalization probes, never trained on).
- **lens** — `logit` (raw unembedding at each layer) vs `tuned` (a lens
  trained on the *base* model's activations; see README's lens-validity note).
- **H1** — does the correct answer emerge at an *earlier layer* after LoRA
  fine-tuning (shift in `first_layer` / `settle_layer`)?
- **H2** — does that shift differ between facts the model already knew
  (existing conditions) and newly introduced facts (synthetic)?


## H1/H2 — does LoRA move the answer earlier, and does it differ by condition?


### Base→LoRA layer shift, logit lens — pooled histogram

**Presents:** distribution of `base − final` for `first_layer` (earliest layer
where the answer is top-1) and `settle_layer` (earliest layer where it stays
top-1 through the output), one histogram each; positive = LoRA made the
answer appear earlier.

**Hypothesis:** H1 (does LoRA shift the answer to an earlier layer?).

**Population:** train prompts, logit lens, pooled across all four conditions
(known/latent/unknown/synthetic), facts with both a base and a final value for
the given metric.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]
    wide_first = sub.pivot_table(index="prompt_idx", columns="variant", values="first_layer")
    wide_settle = sub.pivot_table(index="prompt_idx", columns="variant", values="settle_layer")

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
    for ax, wide, title in ((axes[0], wide_first, "first_layer"),
                             (axes[1], wide_settle, "settle_layer")):
        if not {"base", "final"}.issubset(wide.columns):
            ax.set_visible(False)
            continue
        both = wide[["base", "final"]].dropna()
        if both.empty:
            ax.set_title(f"{title} (no data)")
            continue
        delta = both["base"] - both["final"]
        ax.hist(delta, bins=20, color="#0072B2", alpha=0.8, edgecolor="white")
        ax.axvline(0, color="black", linewidth=0.8, alpha=0.5)
        mean_d, med_d = float(delta.mean()), float(delta.median())
        pct_pos = 100.0 * (delta > 0).mean()
        ax.axvline(mean_d, color="#D55E00", linewidth=1.2, linestyle="--", label=f"mean={mean_d:.1f}")
        ax.axvline(med_d, color="#E69F00", linewidth=1.2, linestyle=":", label=f"median={med_d:.1f}")
        ax.set_xlabel(r"$\Delta$ = base $-$ final (positive = earlier)")
        ax.set_ylabel("Prompts")
        ax.set_title(title)
        ax.legend(fontsize=7, title=f"{pct_pos:.0f}% $\Delta$>0  n={len(delta)}", title_fontsize=7)

    fig.suptitle(f"Base→LoRA layer shift ({lens} lens, pooled)", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "delta_layer_hist", lens=lens)


### Raw layer distributions, logit lens — base vs LoRA overlay

**Presents:** overlaid histograms of raw `first_layer` and `settle_layer`
values for base and final variants (not the delta — the absolute layer each
lands on).

**Hypothesis:** H1, as a sanity check alongside the delta histogram above
(are the two distributions actually shifted, or just differently shaped?).

**Population:** train prompts, logit lens, pooled across all conditions.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
    for ax, col, title in ((axes[0], "first_layer", "first_layer"),
                            (axes[1], "settle_layer", "settle_layer")):
        for variant, color, label in (("base", "#BBBBBB", "Base"), ("final", "#0072B2", "LoRA")):
            vals = sub.loc[sub["variant"] == variant, col].dropna()
            if vals.empty:
                continue
            ax.hist(vals, bins=20, alpha=0.55, color=color, label=label, edgecolor="white")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Prompts")
        ax.set_title(title)
        ax.legend(fontsize=7)

    fig.suptitle(f"Layer distributions ({lens} lens, pooled)", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "layer_hist", lens=lens)


### Where does LoRA add log-probability? — logit lens

**Presents:** mean `answer_logprob(final) − answer_logprob(base)` per layer, a
single pooled curve, with a vertical marker at the median patching
first-flip layer (causal cross-check).

**Hypothesis:** H1 — if LoRA moves knowledge earlier, this curve should turn
positive at an earlier layer than it would pre-LoRA, and the log-prob gain
should concentrate around the causal locus found by patching.

**Population:** train prompts, logit lens, pooled across all conditions.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["variant"].isin(["base", "final"])) &
            (df["prompt_type"] == "train")]

    mean_lp = df.groupby(["variant", "layer"])["answer_logprob"].mean().reset_index()
    base_lp = mean_lp[mean_lp["variant"] == "base"].rename(columns={"answer_logprob": "base_lp"}).drop(columns="variant")
    final_lp = mean_lp[mean_lp["variant"] == "final"].rename(columns={"answer_logprob": "final_lp"}).drop(columns="variant")
    delta = base_lp.merge(final_lp, on="layer")
    delta["delta_lp"] = delta["final_lp"] - delta["base_lp"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    ax.plot(delta["layer"], delta["delta_lp"], color="#0072B2", marker="o",
            markevery=MARKER_EVERY, markersize=4, label="Pooled Δ log-prob")

    med = viz._pooled_median_first_flip(RESULTS_DIR)
    if med is not None:
        ax.axvline(med, color="#D55E00", linewidth=1.0, linestyle=":", label=f"median first-flip={med:.0f}")

    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("Δ mean log-probability\n(LoRA − base)")
    ax.set_title(f"Where does LoRA add log-probability? ({lens})")
    ax.set_xlim(0, int(delta["layer"].max()) if not delta.empty else 24)
    ax.legend(loc="upper left", fontsize=7)

    fig.tight_layout(pad=0.5)
    show_and_save(fig, "delta_logprob", lens=lens)


### Layer-wise accuracy before vs after LoRA — logit lens

**Presents:** grouped bars of % of facts with the correct answer ranked #1 at
each layer, base (pre-LoRA) vs final (post-LoRA).

**Hypothesis:** H1 — accuracy at early/middle layers should rise post-LoRA if
the answer is emerging earlier, not just becoming more confident at the top.

**Population:** train prompts, logit lens, pooled across all conditions.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base = viz._layer_top1_pooled(df, "base", lens)
    final = viz._layer_top1_pooled(df, "final", lens)
    if base.empty or final.empty:
        print("[notebook] missing base or final rows — skipping.")
    else:
        merged = base.merge(final, on="layer", suffixes=("_base", "_final"))
        n_layers = int(merged["layer"].max())
        layers = np.arange(n_layers + 1)
        width = 0.38

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.bar(merged["layer"] - width / 2, merged["pct_top1_base"], width,
               color="#BBBBBB", alpha=0.9, edgecolor="white", linewidth=0.4, label="Before LoRA (base)")
        ax.bar(merged["layer"] + width / 2, merged["pct_top1_final"], width,
               color="#0072B2", alpha=0.9, edgecolor="white", linewidth=0.4, label="After LoRA (final)")
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel("% of facts with correct answer\n(rank-1 at this layer)")
        ax.set_title(f"Layer-wise accuracy — before vs after ({lens})")
        ax.set_xlim(-0.5, n_layers + 0.5)
        ax.set_xticks(layers[::max(1, len(layers) // 8)])
        ymax = max(merged["pct_top1_base"].max(), merged["pct_top1_final"].max())
        ax.set_ylim(0, min(105, ymax * 1.15 + 1))
        ax.legend(loc="upper left", fontsize=7)

        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_top1_combined_comparison", lens=lens)


### Moderator: prior signal vs settle shift — logit lens

**Presents:** scatter of baseline (pre-LoRA) answer log-prob against the
settle-layer shift (`base − final`), with an OLS regression line and
Spearman ρ; also reports agreement with the patching-derived depth when
`trajectory_vs_patching.csv` exists.

**Hypothesis:** H1 refinement — does how strongly the base model already
leans toward the answer (its prior) predict how much earlier LoRA moves it?

**Population:** train prompts, logit lens, facts with a settle layer in both
base and final variants, pooled across conditions.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if traj is None or not path.exists():
    print("[notebook] missing trajectory/layerwise — skipping.")
else:
    data = moderator_data(traj, pd.read_parquet(path), lens=lens)
    if data.empty:
        print("[notebook] empty moderator_data — skipping.")
    else:
        pts = data[["baseline_logprob", "base_settle", "final_settle"]].dropna()
        if len(pts) < 5:
            print("[notebook] too few settle pairs — skipping.")
        else:
            pts = pts.copy()
            pts["y"] = pts["base_settle"] - pts["final_settle"]

            fig, ax = plt.subplots(figsize=(3.5, 2.8))
            ax.scatter(pts["baseline_logprob"], pts["y"], s=12, alpha=0.45, color="#0072B2")

            slope, intercept, r, p, _ = sstats.linregress(pts["baseline_logprob"], pts["y"])
            xs = np.linspace(pts["baseline_logprob"].min(), pts["baseline_logprob"].max(), 50)
            ax.plot(xs, intercept + slope * xs, color="#D55E00", linewidth=1.5)
            rho, p_sp = sstats.spearmanr(pts["baseline_logprob"], pts["y"])
            r2 = r ** 2
            ax.text(0.05, 0.95, f"slope={slope:.2f}\nSpearman ρ={rho:.2f}\np={p_sp:.2g}\nR²={r2:.2f}",
                    transform=ax.transAxes, va="top", fontsize=7,
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

            vs_path = RESULTS_DIR / "trajectory_vs_patching.csv"
            note = ""
            if vs_path.exists():
                vs = pd.read_csv(vs_path)
                if "lens" in vs.columns:
                    vs = vs[vs["lens"] == lens]
                if not vs.empty:
                    pair = vs[["settle_layer", "first_flip_layer"]].dropna()
                    if len(pair) >= 5:
                        rho2, _ = sstats.spearmanr(pair["settle_layer"], pair["first_flip_layer"])
                        gap = float((pair["settle_layer"] - pair["first_flip_layer"]).median())
                        note = f"\n(vs patching: ρ={rho2:.2f}, med gap={gap:.1f})"

            ax.set_xlabel("Baseline answer log-prob")
            ax.set_ylabel(r"Settle shift (base $-$ final)")
            ax.set_title(f"Moderator: prior signal vs settle shift ({lens}){note}")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "moderator_scatter", lens=lens)


### Base→LoRA layer shift, tuned lens — pooled histogram

**Presents:** distribution of `base − final` for `first_layer` (earliest layer
where the answer is top-1) and `settle_layer` (earliest layer where it stays
top-1 through the output), one histogram each; positive = LoRA made the
answer appear earlier.

**Hypothesis:** H1 (does LoRA shift the answer to an earlier layer?).

**Population:** train prompts, tuned lens, pooled across all four conditions
(known/latent/unknown/synthetic), facts with both a base and a final value for
the given metric.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]
    wide_first = sub.pivot_table(index="prompt_idx", columns="variant", values="first_layer")
    wide_settle = sub.pivot_table(index="prompt_idx", columns="variant", values="settle_layer")

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
    for ax, wide, title in ((axes[0], wide_first, "first_layer"),
                             (axes[1], wide_settle, "settle_layer")):
        if not {"base", "final"}.issubset(wide.columns):
            ax.set_visible(False)
            continue
        both = wide[["base", "final"]].dropna()
        if both.empty:
            ax.set_title(f"{title} (no data)")
            continue
        delta = both["base"] - both["final"]
        ax.hist(delta, bins=20, color="#0072B2", alpha=0.8, edgecolor="white")
        ax.axvline(0, color="black", linewidth=0.8, alpha=0.5)
        mean_d, med_d = float(delta.mean()), float(delta.median())
        pct_pos = 100.0 * (delta > 0).mean()
        ax.axvline(mean_d, color="#D55E00", linewidth=1.2, linestyle="--", label=f"mean={mean_d:.1f}")
        ax.axvline(med_d, color="#E69F00", linewidth=1.2, linestyle=":", label=f"median={med_d:.1f}")
        ax.set_xlabel(r"$\Delta$ = base $-$ final (positive = earlier)")
        ax.set_ylabel("Prompts")
        ax.set_title(title)
        ax.legend(fontsize=7, title=f"{pct_pos:.0f}% $\Delta$>0  n={len(delta)}", title_fontsize=7)

    fig.suptitle(f"Base→LoRA layer shift ({lens} lens, pooled)", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "delta_layer_hist", lens=lens)


### Raw layer distributions, tuned lens — base vs LoRA overlay

**Presents:** overlaid histograms of raw `first_layer` and `settle_layer`
values for base and final variants (not the delta — the absolute layer each
lands on).

**Hypothesis:** H1, as a sanity check alongside the delta histogram above
(are the two distributions actually shifted, or just differently shaped?).

**Population:** train prompts, tuned lens, pooled across all conditions.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory data available — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))]

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
    for ax, col, title in ((axes[0], "first_layer", "first_layer"),
                            (axes[1], "settle_layer", "settle_layer")):
        for variant, color, label in (("base", "#BBBBBB", "Base"), ("final", "#0072B2", "LoRA")):
            vals = sub.loc[sub["variant"] == variant, col].dropna()
            if vals.empty:
                continue
            ax.hist(vals, bins=20, alpha=0.55, color=color, label=label, edgecolor="white")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Prompts")
        ax.set_title(title)
        ax.legend(fontsize=7)

    fig.suptitle(f"Layer distributions ({lens} lens, pooled)", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "layer_hist", lens=lens)


### Where does LoRA add log-probability? — tuned lens

**Presents:** mean `answer_logprob(final) − answer_logprob(base)` per layer, a
single pooled curve, with a vertical marker at the median patching
first-flip layer (causal cross-check).

**Hypothesis:** H1 — if LoRA moves knowledge earlier, this curve should turn
positive at an earlier layer than it would pre-LoRA, and the log-prob gain
should concentrate around the causal locus found by patching.

**Population:** train prompts, tuned lens, pooled across all conditions.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["variant"].isin(["base", "final"])) &
            (df["prompt_type"] == "train")]

    mean_lp = df.groupby(["variant", "layer"])["answer_logprob"].mean().reset_index()
    base_lp = mean_lp[mean_lp["variant"] == "base"].rename(columns={"answer_logprob": "base_lp"}).drop(columns="variant")
    final_lp = mean_lp[mean_lp["variant"] == "final"].rename(columns={"answer_logprob": "final_lp"}).drop(columns="variant")
    delta = base_lp.merge(final_lp, on="layer")
    delta["delta_lp"] = delta["final_lp"] - delta["base_lp"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    ax.plot(delta["layer"], delta["delta_lp"], color="#0072B2", marker="o",
            markevery=MARKER_EVERY, markersize=4, label="Pooled Δ log-prob")

    med = viz._pooled_median_first_flip(RESULTS_DIR)
    if med is not None:
        ax.axvline(med, color="#D55E00", linewidth=1.0, linestyle=":", label=f"median first-flip={med:.0f}")

    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("Δ mean log-probability\n(LoRA − base)")
    ax.set_title(f"Where does LoRA add log-probability? ({lens})")
    ax.set_xlim(0, int(delta["layer"].max()) if not delta.empty else 24)
    ax.legend(loc="upper left", fontsize=7)

    fig.tight_layout(pad=0.5)
    show_and_save(fig, "delta_logprob", lens=lens)


### Layer-wise accuracy before vs after LoRA — tuned lens

**Presents:** grouped bars of % of facts with the correct answer ranked #1 at
each layer, base (pre-LoRA) vs final (post-LoRA).

**Hypothesis:** H1 — accuracy at early/middle layers should rise post-LoRA if
the answer is emerging earlier, not just becoming more confident at the top.

**Population:** train prompts, tuned lens, pooled across all conditions.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base = viz._layer_top1_pooled(df, "base", lens)
    final = viz._layer_top1_pooled(df, "final", lens)
    if base.empty or final.empty:
        print("[notebook] missing base or final rows — skipping.")
    else:
        merged = base.merge(final, on="layer", suffixes=("_base", "_final"))
        n_layers = int(merged["layer"].max())
        layers = np.arange(n_layers + 1)
        width = 0.38

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.bar(merged["layer"] - width / 2, merged["pct_top1_base"], width,
               color="#BBBBBB", alpha=0.9, edgecolor="white", linewidth=0.4, label="Before LoRA (base)")
        ax.bar(merged["layer"] + width / 2, merged["pct_top1_final"], width,
               color="#0072B2", alpha=0.9, edgecolor="white", linewidth=0.4, label="After LoRA (final)")
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel("% of facts with correct answer\n(rank-1 at this layer)")
        ax.set_title(f"Layer-wise accuracy — before vs after ({lens})")
        ax.set_xlim(-0.5, n_layers + 0.5)
        ax.set_xticks(layers[::max(1, len(layers) // 8)])
        ymax = max(merged["pct_top1_base"].max(), merged["pct_top1_final"].max())
        ax.set_ylim(0, min(105, ymax * 1.15 + 1))
        ax.legend(loc="upper left", fontsize=7)

        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_top1_combined_comparison", lens=lens)


### Moderator: prior signal vs settle shift — tuned lens

**Presents:** scatter of baseline (pre-LoRA) answer log-prob against the
settle-layer shift (`base − final`), with an OLS regression line and
Spearman ρ; also reports agreement with the patching-derived depth when
`trajectory_vs_patching.csv` exists.

**Hypothesis:** H1 refinement — does how strongly the base model already
leans toward the answer (its prior) predict how much earlier LoRA moves it?

**Population:** train prompts, tuned lens, facts with a settle layer in both
base and final variants, pooled across conditions.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if traj is None or not path.exists():
    print("[notebook] missing trajectory/layerwise — skipping.")
else:
    data = moderator_data(traj, pd.read_parquet(path), lens=lens)
    if data.empty:
        print("[notebook] empty moderator_data — skipping.")
    else:
        pts = data[["baseline_logprob", "base_settle", "final_settle"]].dropna()
        if len(pts) < 5:
            print("[notebook] too few settle pairs — skipping.")
        else:
            pts = pts.copy()
            pts["y"] = pts["base_settle"] - pts["final_settle"]

            fig, ax = plt.subplots(figsize=(3.5, 2.8))
            ax.scatter(pts["baseline_logprob"], pts["y"], s=12, alpha=0.45, color="#0072B2")

            slope, intercept, r, p, _ = sstats.linregress(pts["baseline_logprob"], pts["y"])
            xs = np.linspace(pts["baseline_logprob"].min(), pts["baseline_logprob"].max(), 50)
            ax.plot(xs, intercept + slope * xs, color="#D55E00", linewidth=1.5)
            rho, p_sp = sstats.spearmanr(pts["baseline_logprob"], pts["y"])
            r2 = r ** 2
            ax.text(0.05, 0.95, f"slope={slope:.2f}\nSpearman ρ={rho:.2f}\np={p_sp:.2g}\nR²={r2:.2f}",
                    transform=ax.transAxes, va="top", fontsize=7,
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

            vs_path = RESULTS_DIR / "trajectory_vs_patching.csv"
            note = ""
            if vs_path.exists():
                vs = pd.read_csv(vs_path)
                if "lens" in vs.columns:
                    vs = vs[vs["lens"] == lens]
                if not vs.empty:
                    pair = vs[["settle_layer", "first_flip_layer"]].dropna()
                    if len(pair) >= 5:
                        rho2, _ = sstats.spearmanr(pair["settle_layer"], pair["first_flip_layer"])
                        gap = float((pair["settle_layer"] - pair["first_flip_layer"]).median())
                        note = f"\n(vs patching: ρ={rho2:.2f}, med gap={gap:.1f})"

            ax.set_xlabel("Baseline answer log-prob")
            ax.set_ylabel(r"Settle shift (base $-$ final)")
            ax.set_title(f"Moderator: prior signal vs settle shift ({lens}){note}")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "moderator_scatter", lens=lens)


### Causal locus of the LoRA update — per-condition patching

**Presents:** left panel — boxplot of first-flip layer (earliest layer whose
patched activation flips the base model to the LoRA answer) by condition,
with flip counts annotated; right panel — number of facts flipped vs training
step, per condition.

**Hypothesis:** H2 — activation patching is lens-free, so it's the causal
cross-check on whether known/latent/unknown/synthetic facts get written into
different depths and at different points in training.

**Population:** all conditions, final LoRA checkpoint (left) and every
checkpoint (right); lens-free (patching operates on hidden states directly).


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    summary = df[df["layer"] == -1].copy()

    fig, (ax_box, ax_dyn) = plt.subplots(1, 2, figsize=(6.8, 2.8))

    final = summary[summary["variant"] == "final"].copy()
    present = [c for c in COND_ORDER if c in final["condition"].values]
    n_layers_patch = int(df["layer"][df["layer"] >= 0].max()) if (df["layer"] >= 0).any() else 24
    ax_box.set_ylim(-1, n_layers_patch + 2)

    for i, cond in enumerate(present):
        sub = final[final["condition"] == cond]["first_flip_layer"].dropna()
        n_total = len(final[final["condition"] == cond])
        n_flip = len(sub)
        bp = ax_box.boxplot(sub, positions=[i], widths=0.45, patch_artist=True,
                             medianprops=dict(color="white", linewidth=2),
                             whiskerprops=dict(linewidth=1.2),
                             capprops=dict(linewidth=1.2),
                             flierprops=dict(marker="o", markersize=2.5,
                                             markerfacecolor=COLORS[cond], alpha=0.5))
        bp["boxes"][0].set(facecolor=COLORS[cond], alpha=0.75)
        jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(sub))
        ax_box.scatter(i + jitter, sub, color=COLORS[cond], alpha=0.35, s=8, zorder=3)
        ax_box.text(i, ax_box.get_ylim()[1], f"{n_flip}/{n_total}", ha="center", va="bottom",
                    fontsize=7.5, color=COLORS[cond], fontweight="bold")

    ax_box.set_xticks(list(range(len(present))))
    ax_box.set_xticklabels([CONDITION_LABELS[c] for c in present])
    ax_box.set_ylabel("First-flip layer")
    ax_box.set_title("Causal locus of LoRA update")

    n_cap = int(final[final["condition"] != "known"].groupby("condition").size().max() or 100)
    for cond in [c for c in COND_ORDER if c != "known"]:
        sub = (summary[(summary["condition"] == cond) & (summary["variant"] != "final")]
               .groupby("step")["first_flip_layer"].count().reset_index(name="flipped"))
        final_count = summary[(summary["condition"] == cond) &
                               (summary["variant"] == "final")]["first_flip_layer"].count()
        final_step_val = summary[(summary["variant"] == "final")]["step"].max()
        sub = pd.concat([sub, pd.DataFrame([{"step": final_step_val, "flipped": final_count}])],
                        ignore_index=True).sort_values("step")
        ax_dyn.plot(sub["step"], sub["flipped"], color=COLORS[cond], linestyle=LINE_STYLES[cond],
                    marker=MARKERS[cond], markersize=5, label=CONDITION_LABELS[cond])

    ax_dyn.set_xlabel("Training step")
    ax_dyn.set_ylabel(f"Facts where patching flips\nprediction (out of {n_cap})")
    ax_dyn.set_title("Learning dynamics")
    ax_dyn.set_ylim(0, n_cap)
    ax_dyn.legend()

    fig.tight_layout(pad=0.4, w_pad=1.5)
    show_and_save(fig, "patching")


### Causal locus of the LoRA update — pooled

**Presents:** same two panels as above (first-flip boxplot, flip count vs
step), but pooled across conditions instead of split.

**Hypothesis:** H1's causal counterpart at the aggregate level — same
questions as the pooled log-prob/accuracy figures, but from the patching
(lens-free) evidence.

**Population:** all conditions pooled, final checkpoint (left) and every
checkpoint (right).


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    summary = df[df["layer"] == -1].copy()
    final = summary[summary["variant"] == "final"]
    flips = final["first_flip_layer"].dropna()
    n_total, n_flip = len(final), len(flips)

    fig, (ax_box, ax_dyn) = plt.subplots(1, 2, figsize=(6.8, 2.8))
    n_layers_patch = int(df["layer"][df["layer"] >= 0].max()) if (df["layer"] >= 0).any() else 24
    ax_box.set_ylim(-1, n_layers_patch + 2)

    bp = ax_box.boxplot(flips, positions=[0], widths=0.45, patch_artist=True,
                         medianprops=dict(color="white", linewidth=2))
    bp["boxes"][0].set(facecolor="#0072B2", alpha=0.75)
    jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(flips))
    ax_box.scatter(jitter, flips, color="#0072B2", alpha=0.35, s=8, zorder=3)
    ax_box.text(0, ax_box.get_ylim()[1], f"{n_flip}/{n_total}", ha="center", va="bottom",
                fontsize=8, fontweight="bold")
    ax_box.set_xticks([0])
    ax_box.set_xticklabels(["Pooled"])
    ax_box.set_ylabel("First-flip layer")
    ax_box.set_title("Causal locus (pooled)")

    by_step = (summary[summary["variant"] != "final"]
               .groupby("step")["first_flip_layer"].count().reset_index(name="flipped"))
    final_count = final["first_flip_layer"].count()
    final_step = summary[summary["variant"] == "final"]["step"].max()
    by_step = pd.concat([by_step, pd.DataFrame([{"step": final_step, "flipped": final_count}])],
                        ignore_index=True).sort_values("step")
    ax_dyn.plot(by_step["step"], by_step["flipped"], color="#0072B2", marker="o", markersize=5)
    ax_dyn.set_xlabel("Training step")
    ax_dyn.set_ylabel("Facts flipped (pooled)")
    ax_dyn.set_title("Learning dynamics (pooled)")

    fig.tight_layout(pad=0.4, w_pad=1.5)
    show_and_save(fig, "patching_pooled")


### Patching depth distributions — pooled

**Presents:** (a) histogram of `first_flip_layer` (earliest layer that flips
the prediction when patched); (b) `persistent_flip_layer` — earliest layer
after which the flip holds through to the output.

**Hypothesis:** H1's causal counterpart — is the flip a transient
disturbance or does it persist, and at what depth?

**Population:** final LoRA checkpoint, all conditions pooled, lens-free.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    final_sum = df[(df["layer"] == -1) & (df["variant"] == "final")]
    first = final_sum["first_flip_layer"].dropna()

    persist = viz._persistent_flip_layer(df[df["variant"] == "final"])
    persist_vals = persist["persistent_flip_layer"].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
    axes[0].hist(first, bins=20, color="#0072B2", alpha=0.8, edgecolor="white")
    axes[0].set_xlabel("first_flip_layer")
    axes[0].set_ylabel("Facts")
    axes[0].set_title(f"First flip (n={len(first)})")

    axes[1].hist(persist_vals, bins=20, color="#D55E00", alpha=0.8, edgecolor="white")
    axes[1].set_xlabel("persistent_flip_layer")
    axes[1].set_ylabel("Facts")
    axes[1].set_title(f"Persistent flip (n={len(persist_vals)})")

    fig.suptitle("Patching depth distributions (pooled)", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4, w_pad=1.2)
    show_and_save(fig, "patching_layer_hist")


## Summary tables


### Table 1 — logit-lens results before/after LoRA (pooled)

**Presents:** one row for train prompts and one for held-out paraphrases:
Acc@1 and mean log-prob (base vs LoRA), plus the shift in mean first-layer
(negative = LoRA moved the answer earlier). Rows are `n_prompts`-weighted
averages across conditions.

**Hypothesis:** H1 headline numbers — the scalar summary the layer-shift
figures above visualize.

**Population:** train and paraphrase prompts, logit lens, pooled across all
conditions.


In [ ]:
lens = "logit"
summary_path = RESULTS_DIR / "summary.csv"
if not summary_path.exists():
    print(f"[notebook] {summary_path} not found — skipping.")
else:
    s_all = pd.read_csv(summary_path)
    s = s_all[(s_all["lens"] == lens) & (s_all["variant"].isin(["base", "final"]))]

    def _wavg(frame, col):
        if "n_prompts" in frame.columns and frame["n_prompts"].sum():
            return np.average(frame[col], weights=frame["n_prompts"])
        return frame[col].mean()

    rows = []
    for ptype, label in (("train", "train"), ("paraphrase", "para.")):
        b = s[(s["variant"] == "base") & (s["prompt_type"] == ptype)]
        f = s[(s["variant"] == "final") & (s["prompt_type"] == ptype)]
        if b.empty or f.empty:
            continue
        acc_b, acc_f = _wavg(b, "final_accuracy"), _wavg(f, "final_accuracy")
        lp_b, lp_f = _wavg(b, "mean_final_logprob"), _wavg(f, "mean_final_logprob")
        fl_b, fl_f = _wavg(b, "mean_first_layer"), _wavg(f, "mean_first_layer")
        rows.append({
            "prompts": label, "acc@1_base": acc_b, "acc@1_lora": acc_f,
            "logprob_base": lp_b, "logprob_lora": lp_f,
            "first_layer_shift": fl_f - fl_b,
        })
    table1 = pd.DataFrame(rows).set_index("prompts").round(3)
    display(table1)


### Table 1 — tuned-lens results before/after LoRA (pooled)

**Presents:** one row for train prompts and one for held-out paraphrases:
Acc@1 and mean log-prob (base vs LoRA), plus the shift in mean first-layer
(negative = LoRA moved the answer earlier). Rows are `n_prompts`-weighted
averages across conditions.

**Hypothesis:** H1 headline numbers — the scalar summary the layer-shift
figures above visualize.

**Population:** train and paraphrase prompts, tuned lens, pooled across all
conditions.


In [ ]:
lens = "tuned"
summary_path = RESULTS_DIR / "summary.csv"
if not summary_path.exists():
    print(f"[notebook] {summary_path} not found — skipping.")
else:
    s_all = pd.read_csv(summary_path)
    s = s_all[(s_all["lens"] == lens) & (s_all["variant"].isin(["base", "final"]))]

    def _wavg(frame, col):
        if "n_prompts" in frame.columns and frame["n_prompts"].sum():
            return np.average(frame[col], weights=frame["n_prompts"])
        return frame[col].mean()

    rows = []
    for ptype, label in (("train", "train"), ("paraphrase", "para.")):
        b = s[(s["variant"] == "base") & (s["prompt_type"] == ptype)]
        f = s[(s["variant"] == "final") & (s["prompt_type"] == ptype)]
        if b.empty or f.empty:
            continue
        acc_b, acc_f = _wavg(b, "final_accuracy"), _wavg(f, "final_accuracy")
        lp_b, lp_f = _wavg(b, "mean_final_logprob"), _wavg(f, "mean_final_logprob")
        fl_b, fl_f = _wavg(b, "mean_first_layer"), _wavg(f, "mean_first_layer")
        rows.append({
            "prompts": label, "acc@1_base": acc_b, "acc@1_lora": acc_f,
            "logprob_base": lp_b, "logprob_lora": lp_f,
            "first_layer_shift": fl_f - fl_b,
        })
    table1 = pd.DataFrame(rows).set_index("prompts").round(3)
    display(table1)


### Table 1b — logit-lens vs tuned-lens agreement (final checkpoint)

**Presents:** Acc@1 and mean first-layer per condition, logit lens vs tuned
lens, at the final LoRA checkpoint on training prompts.

**Hypothesis:** the lens-validity check the README calls out — the tuned
lens is fit on the base model, so agreement here is what licenses trusting
either lens's H1/H2 conclusions.

**Population:** train prompts, final checkpoint, both lenses required, all
conditions.


In [ ]:
summary_path = RESULTS_DIR / "summary.csv"
if not summary_path.exists():
    print(f"[notebook] {summary_path} not found — skipping.")
else:
    s_all = pd.read_csv(summary_path)
    cmp_df = s_all[(s_all["variant"] == "final") & (s_all["prompt_type"] == "train")]
    if "tuned" not in cmp_df["lens"].unique():
        print("[notebook] no tuned-lens rows — skipping.")
    else:
        rows = []
        for cond in COND_ORDER:
            lg = cmp_df[(cmp_df["condition"] == cond) & (cmp_df["lens"] == "logit")]
            tn = cmp_df[(cmp_df["condition"] == cond) & (cmp_df["lens"] == "tuned")]
            if lg.empty or tn.empty:
                continue
            lg, tn = lg.iloc[0], tn.iloc[0]
            rows.append({
                "condition": CONDITION_LABELS[cond],
                "acc@1_logit": lg["final_accuracy"], "acc@1_tuned": tn["final_accuracy"],
                "first_layer_logit": lg["mean_first_layer"], "first_layer_tuned": tn["mean_first_layer"],
            })
        table1b = pd.DataFrame(rows).set_index("condition").round(3)
        display(table1b)


### Table 2 — activation patching at the final checkpoint (pooled)

**Presents:** how many facts flip under patching, and the mean/median
first-flip layer, pooled across conditions.

**Hypothesis:** H1's causal scalar summary (companion to the patching
figures above).

**Population:** final LoRA checkpoint, all conditions pooled, lens-free.


In [ ]:
patch_path = RESULTS_DIR / "patching.csv"
if not patch_path.exists():
    print(f"[notebook] {patch_path} not found — skipping.")
else:
    p = pd.read_csv(patch_path)
    p = p[(p["layer"] == -1) & (p["variant"] == "final")]
    n_flipped = int(p["first_flip_layer"].notna().sum())
    n_total = len(p)
    table2 = pd.DataFrame([{
        "set": "pooled",
        "flipped": f"{n_flipped}/{n_total}",
        "mean_first_flip": p["first_flip_layer"].mean(),
        "median_first_flip": p["first_flip_layer"].median(),
    }]).set_index("set").round(2)
    display(table2)


### Table — trajectory class shares, logit lens (pooled)

**Presents:** the fraction of facts in each trajectory class (`never` /
`transient` / `late_only` / `persistent`), base vs final variant.

**Hypothesis:** H1 — a persistent-class share that grows post-LoRA at the
expense of `never`/`transient` is the trajectory-level signature of the
answer emerging and *staying* correct earlier.

**Population:** train prompts, logit lens, pooled across all conditions.
Requires the `trajectory` pipeline stage to have written
`trajectory_summary.csv` for this run.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "trajectory_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found for this run — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train") & (s["variant"].isin(["base", "final"]))]
    if s.empty:
        print("[notebook] no rows for this lens — skipping.")
    else:
        rows = []
        for variant in ("base", "final"):
            sub = s[s["variant"] == variant]
            totals = {c: sub[c].sum() if c in sub.columns else 0 for c in CLASS_ORDER}
            n = sum(totals.values()) or 1
            rows.append({"variant": variant, **{c: totals[c] / n for c in CLASS_ORDER}})
        table = pd.DataFrame(rows).set_index("variant").round(3)
        display(table)


### Table — base→final trajectory transitions, logit lens, by condition

**Presents:** one confusion-style matrix per condition (base class × final
class fact counts), plus a McNemar test comparing "ever correct" vs
"survives to the end" where available.

**Hypothesis:** H2 — do known/latent/unknown/synthetic facts transition
between trajectory classes differently (e.g. does `transient → persistent`
happen more for known facts than synthetic ones)?

**Population:** train prompts, logit lens, split per condition. Requires
`trajectory_transitions.csv` for this run.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "trajectory_transitions.csv"
if not path.exists():
    print(f"[notebook] {path} not found for this run — skipping.")
else:
    t = pd.read_csv(path)
    if "lens" in t.columns:
        t = t[t["lens"] == lens]
    shown = False
    for cond in COND_ORDER:
        mat = t[(t["condition"] == cond) & (t["base_class"] != "ANY")]
        if mat.empty:
            continue
        pivot = mat.pivot_table(index="base_class", columns="final_class", values="n",
                                 aggfunc="sum").reindex(index=CLASS_ORDER, columns=CLASS_ORDER).fillna(0).astype(int)
        print(f"-- {CONDITION_LABELS[cond]} ({lens}) --")
        mcn = t[(t["condition"] == cond) & (t["base_class"] == "ANY")]
        if not mcn.empty:
            row = mcn.iloc[0]
            print(f"McNemar ANY→SURVIVES: n01={int(row.get('n01', 0))}, "
                  f"n10={int(row.get('n10', 0))}, p={row.get('p_value', float('nan')):.3g}")
        display(pivot)
        shown = True
    if not shown:
        print("[notebook] no transition rows for this lens — skipping.")


### Table — trajectory class shares, tuned lens (pooled)

**Presents:** the fraction of facts in each trajectory class (`never` /
`transient` / `late_only` / `persistent`), base vs final variant.

**Hypothesis:** H1 — a persistent-class share that grows post-LoRA at the
expense of `never`/`transient` is the trajectory-level signature of the
answer emerging and *staying* correct earlier.

**Population:** train prompts, tuned lens, pooled across all conditions.
Requires the `trajectory` pipeline stage to have written
`trajectory_summary.csv` for this run.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "trajectory_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found for this run — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train") & (s["variant"].isin(["base", "final"]))]
    if s.empty:
        print("[notebook] no rows for this lens — skipping.")
    else:
        rows = []
        for variant in ("base", "final"):
            sub = s[s["variant"] == variant]
            totals = {c: sub[c].sum() if c in sub.columns else 0 for c in CLASS_ORDER}
            n = sum(totals.values()) or 1
            rows.append({"variant": variant, **{c: totals[c] / n for c in CLASS_ORDER}})
        table = pd.DataFrame(rows).set_index("variant").round(3)
        display(table)


### Table — base→final trajectory transitions, tuned lens, by condition

**Presents:** one confusion-style matrix per condition (base class × final
class fact counts), plus a McNemar test comparing "ever correct" vs
"survives to the end" where available.

**Hypothesis:** H2 — do known/latent/unknown/synthetic facts transition
between trajectory classes differently (e.g. does `transient → persistent`
happen more for known facts than synthetic ones)?

**Population:** train prompts, tuned lens, split per condition. Requires
`trajectory_transitions.csv` for this run.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "trajectory_transitions.csv"
if not path.exists():
    print(f"[notebook] {path} not found for this run — skipping.")
else:
    t = pd.read_csv(path)
    if "lens" in t.columns:
        t = t[t["lens"] == lens]
    shown = False
    for cond in COND_ORDER:
        mat = t[(t["condition"] == cond) & (t["base_class"] != "ANY")]
        if mat.empty:
            continue
        pivot = mat.pivot_table(index="base_class", columns="final_class", values="n",
                                 aggfunc="sum").reindex(index=CLASS_ORDER, columns=CLASS_ORDER).fillna(0).astype(int)
        print(f"-- {CONDITION_LABELS[cond]} ({lens}) --")
        mcn = t[(t["condition"] == cond) & (t["base_class"] == "ANY")]
        if not mcn.empty:
            row = mcn.iloc[0]
            print(f"McNemar ANY→SURVIVES: n01={int(row.get('n01', 0))}, "
                  f"n10={int(row.get('n10', 0))}, p={row.get('p_value', float('nan')):.3g}")
        display(pivot)
        shown = True
    if not shown:
        print("[notebook] no transition rows for this lens — skipping.")


## Graphs 1–10 — layer dynamics, generalization, and rank ablation


### Graph 1 — layer top-1 on previously *detectable* facts, logit lens

**Presents:** % top-1 per layer, base vs LoRA, restricted to facts the base
model could already surface (rank-1) at *some* layer before fine-tuning.

**Hypothesis:** H1 on the cleanest subset — for facts the base model already
"has" somewhere in its forward pass, does LoRA pull that answer to an
earlier layer, or just sharpen it at the same depth?

**Population:** train prompts, logit lens; facts where the base model's
`first_layer` is non-null at any layer ("lens-detectable").


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
traj = viz._load_traj(RESULTS_DIR)
if not path.exists() or traj is None:
    print("[notebook] missing data — skipping.")
else:
    idxs = viz.detectable_prompt_idxs(traj, lens)
    if not idxs:
        print("[notebook] no detectable prompts — skipping.")
    else:
        df = pd.read_parquet(path)
        base = viz._layer_top1_pooled(df, "base", lens, prompt_idxs=idxs)
        final = viz._layer_top1_pooled(df, "final", lens, prompt_idxs=idxs)

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.plot(base["layer"], base["pct_top1"], color="#BBBBBB", linestyle="--",
                label="Base", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.plot(final["layer"], final["pct_top1"], color="#0072B2",
                label="LoRA", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel("% top-1 correct")
        ax.set_title(f"Detectable facts (n={len(idxs)}, {lens})")
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_top1_detectable", lens=lens)


### Graph 2a — layer top-1 on all facts, logit lens (pooled)

**Presents:** % top-1 per layer, base vs LoRA, over every train fact
regardless of condition.

**Hypothesis:** H1 headline curve at the full-population level (the pooled
counterpart to Graph 1's detectable-only subset).

**Population:** train prompts, logit lens, all conditions pooled.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base = viz._layer_top1_pooled(df, "base", lens)
    final = viz._layer_top1_pooled(df, "final", lens)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    ax.plot(base["layer"], base["pct_top1"], color="#BBBBBB", linestyle="--",
            label="Base", marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.plot(final["layer"], final["pct_top1"], color="#0072B2",
            label="LoRA", marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("% top-1 correct")
    ax.set_title(f"All facts ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_all_facts", lens=lens)


### Graph 2b — layer top-1 on all facts by condition, logit lens

**Presents:** the same curve as Graph 2a, split by condition (base dashed,
LoRA solid, one color per condition).

**Hypothesis:** H2 — the per-condition breakdown of the headline H1 curve:
does known/latent/unknown/synthetic each get pulled earlier by a different
amount?

**Population:** train prompts, logit lens, one curve per condition.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base_c = viz._layer_top1_curve(df, "base", lens)
    final_c = viz._layer_top1_curve(df, "final", lens)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cond in COND_ORDER:
        b = base_c[base_c["condition"] == cond].sort_values("layer")
        f = final_c[final_c["condition"] == cond].sort_values("layer")
        if not b.empty:
            ax.plot(b["layer"], b["pct_top1"], color=COLORS[cond], linestyle="--", linewidth=1.2, alpha=0.7)
        if not f.empty:
            ax.plot(f["layer"], f["pct_top1"], color=COLORS[cond], linestyle=LINE_STYLES[cond],
                    marker=MARKERS[cond], markevery=MARKER_EVERY, markersize=3, label=CONDITION_LABELS[cond])
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("% top-1 correct")
    ax.set_title(f"All facts by condition ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_all_facts_by_condition", lens=lens)


### Graph 3a — train vs paraphrase Acc@1, logit lens (pooled)

**Presents:** two bars — mean final-checkpoint Acc@1 on train prompts vs
held-out paraphrases (averaged over conditions).

**Hypothesis:** generalization check for H1/H2 — did LoRA memorize the exact
training phrasing, or does the earlier-layer effect transfer to a reworded
prompt for the same fact?

**Population:** final checkpoint, logit lens, train and paraphrase prompts,
pooled across conditions.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final")]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for i, ptype in enumerate(["train", "paraphrase"]):
        vals = s[s["prompt_type"] == ptype]["final_accuracy"]
        if vals.empty:
            continue
        ax.bar(i, vals.mean(), color="#0072B2" if ptype == "train" else "#56B4E9", alpha=0.85, label=ptype)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Train", "Paraphrase"])
    ax.set_ylabel("Acc@1 (mean over conditions)")
    ax.set_title(f"Train vs paraphrase Acc@1 ({lens})")
    ax.set_ylim(0, 1.05)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "final_vs_paraphrase", lens=lens)


### Graph 3b — train vs paraphrase Acc@1 by condition, logit lens

**Presents:** grouped bars (train vs paraphrase) per condition.

**Hypothesis:** H2's generalization angle — does the train→paraphrase gap
differ between known/latent/unknown facts and synthetic ones?

**Population:** final checkpoint, logit lens, one pair of bars per
condition.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final")]
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    x = np.arange(len(COND_ORDER))
    width = 0.35
    for offset, ptype, color in ((-width / 2, "train", "#0072B2"), (width / 2, "paraphrase", "#56B4E9")):
        vals = []
        for cond in COND_ORDER:
            row = s[(s["condition"] == cond) & (s["prompt_type"] == ptype)]
            vals.append(float(row["final_accuracy"].iloc[0]) if not row.empty else 0.0)
        ax.bar(x + offset, vals, width, color=color, alpha=0.85, label=ptype)
    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in COND_ORDER], fontsize=7)
    ax.set_ylabel("Acc@1")
    ax.set_title(f"Train vs paraphrase by condition ({lens})")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "final_vs_paraphrase_by_condition", lens=lens)


### Graph 4 — paraphrase layers after LoRA, logit lens (pooled)

**Presents:** layer-wise % top-1 on paraphrase prompts, restricted to facts
that were correct on train both before and after LoRA, wrong on at least one
paraphrase before LoRA, and correct on the paraphrase after LoRA — i.e.
facts where LoRA specifically fixed the generalization gap.

**Hypothesis:** H1 on the "LoRA taught genuine generalization" subset — for
those facts, at what layer does the now-correct paraphrase answer emerge?

**Population:** train+paraphrase prompts, logit lens, final checkpoint,
facts matching the filter above (pooled across conditions; typically a small
n).


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if traj is None or not path.exists():
    print("[notebook] missing data — skipping.")
else:
    t = traj[(traj["lens"] == lens) & (traj["variant"].isin(["base", "final"]))]
    train = t[t["prompt_type"] == "train"]
    para = t[t["prompt_type"] == "paraphrase"]
    if train.empty or para.empty:
        print("[notebook] no paraphrase rows — skipping.")
    else:
        tr = train.set_index(["fact_id", "variant"])
        pr = para.set_index(["fact_id", "variant"])
        base_train_ok = set(tr.loc[(slice(None), "base"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))
        final_train_ok = set(tr.loc[(slice(None), "final"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))
        base_para = pr.loc[(slice(None), "base"), :]
        base_para_bad = set(base_para.loc[
            ~base_para["traj_class"].isin(["late_only", "persistent"])].index.get_level_values(0))
        final_para_ok = set(pr.loc[(slice(None), "final"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))

        facts = base_train_ok & base_para_bad & final_para_ok & final_train_ok
        if not facts:
            print("[notebook] filter matched 0 facts — skipping.")
        else:
            para_final = para[(para["variant"] == "final") & (para["fact_id"].isin(facts))]
            idxs = set(para_final["prompt_idx"])
            df = pd.read_parquet(path)
            curve = viz._layer_top1_pooled(df, "final", lens, prompt_type="paraphrase", prompt_idxs=idxs)

            fig, ax = plt.subplots(figsize=(3.5, 2.8))
            ax.plot(curve["layer"], curve["pct_top1"], color="#0072B2",
                    marker="o", markevery=MARKER_EVERY, markersize=3)
            ax.set_xlabel("Transformer layer")
            ax.set_ylabel("% top-1 (paraphrase, post-LoRA)")
            ax.set_title(f"Paraphrase layers after LoRA (n_facts={len(facts)}, {lens})")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "paraphrase_layers_after_lora", lens=lens)


### Graph 5a — paraphrase layer delay, logit lens (pooled)

**Presents:** histogram of `Δ = L_para − L_orig` (earliest layer on the
paraphrase minus on the original train prompt, post-LoRA); facts never
correct on either side are counted separately and excluded from the
histogram.

**Hypothesis:** H1's generalization-depth counterpart — even when LoRA
generalizes to a paraphrase, does it take extra layers to get there?

**Population:** final checkpoint, logit lens, facts with a first_layer on
both train and (earliest) paraphrase, pooled across conditions.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["variant"] == "final")]
    train = sub[sub["prompt_type"] == "train"][["fact_id", "condition", "first_layer", "traj_class"]].rename(
        columns={"first_layer": "L_orig", "traj_class": "cls_orig"})
    para = (sub[sub["prompt_type"] == "paraphrase"]
            .groupby(["fact_id", "condition"], as_index=False)
            .agg(L_para=("first_layer", "min"),
                 cls_para=("traj_class", lambda s: "never" if (s == "never").all() else "ok")))
    joined = train.merge(para, on=["fact_id", "condition"], how="inner")
    never = joined[(joined["L_orig"].isna()) | (joined["L_para"].isna())]
    both = joined[joined["L_orig"].notna() & joined["L_para"].notna()].copy()
    both["delta"] = both["L_para"] - both["L_orig"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    if not both.empty:
        ax.hist(both["delta"], bins=20, color="#0072B2", alpha=0.8, edgecolor="white")
        ax.axvline(both["delta"].mean(), color="#D55E00", linestyle="--", label=f"mean={both['delta'].mean():.1f}")
    ax.set_xlabel(r"$\Delta = L_{para} - L_{orig}$")
    ax.set_ylabel("Facts")
    ax.set_title(f"Paraphrase layer delay ({lens})\nn={len(both)}, never-correct={len(never)}")
    if not both.empty:
        ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "paraphrase_layer_delay", lens=lens)


### Graph 5b — paraphrase layer delay by condition, logit lens

**Presents:** boxplots of the same `Δ = L_para − L_orig`, split by
condition.

**Hypothesis:** H2's version of Graph 5a — does the generalization-depth
penalty differ between known/latent/unknown facts and synthetic ones?

**Population:** final checkpoint, logit lens, same facts as Graph 5a, split
per condition.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["variant"] == "final")]
    train = sub[sub["prompt_type"] == "train"][["fact_id", "condition", "first_layer", "traj_class"]].rename(
        columns={"first_layer": "L_orig", "traj_class": "cls_orig"})
    para = (sub[sub["prompt_type"] == "paraphrase"]
            .groupby(["fact_id", "condition"], as_index=False)
            .agg(L_para=("first_layer", "min"),
                 cls_para=("traj_class", lambda s: "never" if (s == "never").all() else "ok")))
    joined = train.merge(para, on=["fact_id", "condition"], how="inner")
    both = joined[joined["L_orig"].notna() & joined["L_para"].notna()].copy()
    both["delta"] = both["L_para"] - both["L_orig"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    data, labels = [], []
    for cond in COND_ORDER:
        vals = both.loc[both["condition"] == cond, "delta"]
        if len(vals):
            data.append(vals)
            labels.append(CONDITION_LABELS[cond])
    if data:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True)
        for patch, cond in zip(bp["boxes"], [c for c in COND_ORDER if len(both[both["condition"] == c])]):
            patch.set_facecolor(COLORS[cond])
            patch.set_alpha(0.75)
    ax.set_ylabel(r"$\Delta = L_{para} - L_{orig}$")
    ax.set_title(f"Paraphrase delay by condition ({lens})")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "paraphrase_layer_delay_by_condition", lens=lens)


### Graph 6a — fact-state transitions, logit lens (pooled Sankey)

**Presents:** an alluvial diagram of facts moving between three collapsed
states — `never` / `transient` / `survives` (late_only ∪ persistent) — from
base to final.

**Hypothesis:** H1 as a state-transition story: mass flowing from
`never`/`transient` into `survives` is the qualitative signature of LoRA
making facts stick.

**Population:** train prompts, logit lens, all conditions pooled.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])

    def _flows(frame):
        wide = frame.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        if wide.empty or not {"base", "final"}.issubset(wide.columns):
            return {}
        counts = wide.groupby(["base", "final"]).size()
        return {(a, b): int(n) for (a, b), n in counts.items()}

    fig, ax = plt.subplots(figsize=(4.5, 3.0))
    viz._draw_sankey(ax, _flows(sub))
    ax.set_title(f"Fact-state transitions ({lens}, pooled)")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "fact_state_sankey", lens=lens)


### Graph 6b — fact-state transitions by condition, logit lens

**Presents:** one Sankey panel per condition (same never/transient/survives
states).

**Hypothesis:** H2's version of Graph 6a — do known/latent/unknown/synthetic
facts show different transition patterns?

**Population:** train prompts, logit lens, split per condition.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])

    def _flows(frame):
        wide = frame.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        if wide.empty or not {"base", "final"}.issubset(wide.columns):
            return {}
        counts = wide.groupby(["base", "final"]).size()
        return {(a, b): int(n) for (a, b), n in counts.items()}

    present = [c for c in COND_ORDER if c in sub["condition"].values]
    if not present:
        print("[notebook] no conditions present — skipping.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(3.2 * len(present), 3.0))
        if len(present) == 1:
            axes = [axes]
        for ax, cond in zip(axes, present):
            viz._draw_sankey(ax, _flows(sub[sub["condition"] == cond]))
            ax.set_title(CONDITION_LABELS[cond])
        fig.suptitle(f"Fact-state transitions by condition ({lens})", fontsize=10, y=1.02)
        fig.tight_layout(pad=0.4)
        show_and_save(fig, "fact_state_sankey_by_condition", lens=lens)


### Graph 7 — accuracy vs LoRA rank, logit lens

**Presents:** Acc@1 vs LoRA rank $r$ (rank 0 = base), one line per condition,
from the rank-ablation sweep.

**Hypothesis:** H2's capacity angle — does adapter rank matter differently
for known vs synthetic facts (e.g. does synthetic need higher rank to reach
the same accuracy)?

**Population:** train prompts, logit lens, `rank_ablation_summary.csv`
(re-trained adapters at each swept rank).


In [ ]:
lens = "logit"
path = RESULTS_DIR / "rank_ablation_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cond in COND_ORDER:
        sub = df[df["condition"] == cond].sort_values("rank")
        if sub.empty:
            continue
        ax.plot(sub["rank"], sub["final_accuracy"], color=COLORS[cond], linestyle=LINE_STYLES[cond],
                marker=MARKERS[cond], markersize=4, label=CONDITION_LABELS[cond])
    ax.set_xlabel("LoRA rank $r$ (0 = base)")
    ax.set_ylabel("Acc@1")
    ax.set_title(f"Accuracy vs rank ({lens})")
    ax.set_ylim(-0.03, 1.03)
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "rank_final_accuracy", lens=lens)


### Graph 8 — layer-wise accuracy by rank, faceted by condition, logit lens

**Presents:** one subplot per condition; within each, a layer-wise % top-1
curve per swept rank (viridis colormap, low→high rank).

**Hypothesis:** H1+H2 combined — at which layer does higher rank start to
help, and does that layer/threshold shift by condition?

**Population:** train prompts, logit lens, `rank_ablation_layerwise.parquet`.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "rank_ablation_layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]
    present = [c for c in COND_ORDER if c in df["condition"].values]
    if not present:
        print("[notebook] no conditions present — skipping.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(3.0 * len(present), 2.8), sharey=True)
        if len(present) == 1:
            axes = [axes]
        ranks = sorted(df["rank"].unique()) if "rank" in df.columns else sorted(df["variant"].unique())
        cmap = plt.cm.viridis(np.linspace(0.2, 0.9, len(ranks)))

        for ax, cond in zip(axes, present):
            sub = df[df["condition"] == cond]
            for color, rank in zip(cmap, ranks):
                rsub = sub[sub["rank"] == rank] if "rank" in sub.columns else sub[sub["variant"] == rank]
                curve = (rsub.groupby("layer")["in_top_1"].mean() * 100).reset_index()
                if curve.empty:
                    continue
                ax.plot(curve["layer"], curve["in_top_1"], color=color, label=str(rank), linewidth=1.2)
            ax.set_title(CONDITION_LABELS[cond])
            ax.set_xlabel("Layer")
        axes[0].set_ylabel("% top-1")
        axes[-1].legend(fontsize=6, title="rank", title_fontsize=6)
        fig.suptitle(f"Rank ablation layer accuracy ({lens})", fontsize=10, y=1.02)
        fig.tight_layout(pad=0.4)
        show_and_save(fig, "rank_layer_accuracy", lens=lens)


### Graph 9 — ΔAcc on detectable subset, logit lens

**Presents:** `final − base` % top-1 per layer, restricted to the Graph
1 detectable subset (facts the base model could surface at some layer).

**Hypothesis:** H1 as a direct per-layer effect size on the cleanest
population — where exactly does LoRA add accuracy?

**Population:** train prompts, logit lens, lens-detectable facts (same
filter as Graph 1).


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
traj = viz._load_traj(RESULTS_DIR)
if not path.exists() or traj is None:
    print("[notebook] missing data — skipping.")
else:
    idxs = viz.detectable_prompt_idxs(traj, lens)
    if not idxs:
        print("[notebook] no detectable prompts — skipping.")
    else:
        df = pd.read_parquet(path)
        base = viz._layer_top1_pooled(df, "base", lens, prompt_idxs=idxs)
        final = viz._layer_top1_pooled(df, "final", lens, prompt_idxs=idxs)
        merged = base.merge(final, on="layer", suffixes=("_base", "_final"))
        merged["delta"] = merged["pct_top1_final"] - merged["pct_top1_base"]

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.plot(merged["layer"], merged["delta"], color="#0072B2", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
        ax.set_title(f"ΔAcc on detectable subset (n={len(idxs)}, {lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "delta_detectable", lens=lens)


### Graph 10a — ΔAcc by relation, logit lens

**Presents:** bar chart of `final − base` Acc@1 at the last layer, one bar
per CounterFact relation, sorted descending.

**Hypothesis:** H2's relation-level breakdown — is the LoRA effect uniform
across relation types, or concentrated in a few?

**Population:** train prompts, logit lens, last layer, joined against
`conditions.parquet` for the `relation` field.


In [ ]:
lens = "logit"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    n_layers = int(df["layer"].max())
    final_layer = df[(df["lens"] == lens) & (df["prompt_type"] == "train") &
                      (df["layer"] == n_layers) & (df["variant"].isin(["base", "final"]))]
    joined = final_layer.merge(conds[["fact_id", "relation"]], on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] relation join empty — skipping.")
    else:
        acc = joined.groupby(["relation", "variant"])["in_top_1"].mean().unstack("variant")
        if not {"base", "final"}.issubset(acc.columns):
            print("[notebook] missing base/final for relations — skipping.")
        else:
            acc["delta"] = acc["final"] - acc["base"]
            acc = acc.sort_values("delta", ascending=False)

            fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(acc)), 3.0))
            ax.bar(range(len(acc)), acc["delta"] * 100, color="#0072B2", alpha=0.85)
            ax.set_xticks(range(len(acc)))
            ax.set_xticklabels(acc.index, rotation=90, fontsize=6)
            ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
            ax.set_title(f"ΔAcc by relation ({lens})")
            ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "delta_by_relation", lens=lens)


### Graph 10b — ΔAcc heatmap by relation × layer, logit lens

**Presents:** heatmap of `final − base` % top-1, relations on the y-axis,
layers on the x-axis, diverging red/blue colormap.

**Hypothesis:** H2's fine-grained view of Graph 10a — for relations with a
big net effect, at which layer does that effect actually appear?

**Population:** train prompts, logit lens, every layer, joined against
`conditions.parquet` for `relation`.


In [ ]:
lens = "logit"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    lw = df[(df["lens"] == lens) & (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    lw = lw.merge(conds[["fact_id", "relation"]], on="fact_id", how="inner")
    if lw.empty:
        print("[notebook] relation join empty — skipping.")
    else:
        pivot = lw.groupby(["relation", "variant", "layer"])["in_top_1"].mean().unstack("variant")
        if not {"base", "final"}.issubset(pivot.columns):
            print("[notebook] missing base/final for relations — skipping.")
        else:
            pivot["delta"] = (pivot["final"] - pivot["base"]) * 100
            heat = pivot["delta"].unstack("layer")
            fig, ax = plt.subplots(figsize=(6.5, max(2.5, 0.22 * len(heat))))
            im = ax.imshow(heat.values, aspect="auto", cmap="RdBu_r",
                            vmin=-np.nanmax(np.abs(heat.values)), vmax=np.nanmax(np.abs(heat.values)))
            ax.set_yticks(range(len(heat)))
            ax.set_yticklabels(heat.index, fontsize=6)
            ax.set_xlabel("Layer")
            ax.set_title(f"ΔAcc heatmap by relation × layer ({lens})")
            fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Δ pp")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "delta_by_relation_heatmap", lens=lens)


### Graph 1 — layer top-1 on previously *detectable* facts, tuned lens

**Presents:** % top-1 per layer, base vs LoRA, restricted to facts the base
model could already surface (rank-1) at *some* layer before fine-tuning.

**Hypothesis:** H1 on the cleanest subset — for facts the base model already
"has" somewhere in its forward pass, does LoRA pull that answer to an
earlier layer, or just sharpen it at the same depth?

**Population:** train prompts, tuned lens; facts where the base model's
`first_layer` is non-null at any layer ("lens-detectable").


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
traj = viz._load_traj(RESULTS_DIR)
if not path.exists() or traj is None:
    print("[notebook] missing data — skipping.")
else:
    idxs = viz.detectable_prompt_idxs(traj, lens)
    if not idxs:
        print("[notebook] no detectable prompts — skipping.")
    else:
        df = pd.read_parquet(path)
        base = viz._layer_top1_pooled(df, "base", lens, prompt_idxs=idxs)
        final = viz._layer_top1_pooled(df, "final", lens, prompt_idxs=idxs)

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.plot(base["layer"], base["pct_top1"], color="#BBBBBB", linestyle="--",
                label="Base", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.plot(final["layer"], final["pct_top1"], color="#0072B2",
                label="LoRA", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel("% top-1 correct")
        ax.set_title(f"Detectable facts (n={len(idxs)}, {lens})")
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_top1_detectable", lens=lens)


### Graph 2a — layer top-1 on all facts, tuned lens (pooled)

**Presents:** % top-1 per layer, base vs LoRA, over every train fact
regardless of condition.

**Hypothesis:** H1 headline curve at the full-population level (the pooled
counterpart to Graph 1's detectable-only subset).

**Population:** train prompts, tuned lens, all conditions pooled.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base = viz._layer_top1_pooled(df, "base", lens)
    final = viz._layer_top1_pooled(df, "final", lens)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    ax.plot(base["layer"], base["pct_top1"], color="#BBBBBB", linestyle="--",
            label="Base", marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.plot(final["layer"], final["pct_top1"], color="#0072B2",
            label="LoRA", marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("% top-1 correct")
    ax.set_title(f"All facts ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_all_facts", lens=lens)


### Graph 2b — layer top-1 on all facts by condition, tuned lens

**Presents:** the same curve as Graph 2a, split by condition (base dashed,
LoRA solid, one color per condition).

**Hypothesis:** H2 — the per-condition breakdown of the headline H1 curve:
does known/latent/unknown/synthetic each get pulled earlier by a different
amount?

**Population:** train prompts, tuned lens, one curve per condition.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    base_c = viz._layer_top1_curve(df, "base", lens)
    final_c = viz._layer_top1_curve(df, "final", lens)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cond in COND_ORDER:
        b = base_c[base_c["condition"] == cond].sort_values("layer")
        f = final_c[final_c["condition"] == cond].sort_values("layer")
        if not b.empty:
            ax.plot(b["layer"], b["pct_top1"], color=COLORS[cond], linestyle="--", linewidth=1.2, alpha=0.7)
        if not f.empty:
            ax.plot(f["layer"], f["pct_top1"], color=COLORS[cond], linestyle=LINE_STYLES[cond],
                    marker=MARKERS[cond], markevery=MARKER_EVERY, markersize=3, label=CONDITION_LABELS[cond])
    ax.set_xlabel("Transformer layer")
    ax.set_ylabel("% top-1 correct")
    ax.set_title(f"All facts by condition ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_all_facts_by_condition", lens=lens)


### Graph 3a — train vs paraphrase Acc@1, tuned lens (pooled)

**Presents:** two bars — mean final-checkpoint Acc@1 on train prompts vs
held-out paraphrases (averaged over conditions).

**Hypothesis:** generalization check for H1/H2 — did LoRA memorize the exact
training phrasing, or does the earlier-layer effect transfer to a reworded
prompt for the same fact?

**Population:** final checkpoint, tuned lens, train and paraphrase prompts,
pooled across conditions.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final")]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for i, ptype in enumerate(["train", "paraphrase"]):
        vals = s[s["prompt_type"] == ptype]["final_accuracy"]
        if vals.empty:
            continue
        ax.bar(i, vals.mean(), color="#0072B2" if ptype == "train" else "#56B4E9", alpha=0.85, label=ptype)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Train", "Paraphrase"])
    ax.set_ylabel("Acc@1 (mean over conditions)")
    ax.set_title(f"Train vs paraphrase Acc@1 ({lens})")
    ax.set_ylim(0, 1.05)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "final_vs_paraphrase", lens=lens)


### Graph 3b — train vs paraphrase Acc@1 by condition, tuned lens

**Presents:** grouped bars (train vs paraphrase) per condition.

**Hypothesis:** H2's generalization angle — does the train→paraphrase gap
differ between known/latent/unknown facts and synthetic ones?

**Population:** final checkpoint, tuned lens, one pair of bars per
condition.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final")]
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    x = np.arange(len(COND_ORDER))
    width = 0.35
    for offset, ptype, color in ((-width / 2, "train", "#0072B2"), (width / 2, "paraphrase", "#56B4E9")):
        vals = []
        for cond in COND_ORDER:
            row = s[(s["condition"] == cond) & (s["prompt_type"] == ptype)]
            vals.append(float(row["final_accuracy"].iloc[0]) if not row.empty else 0.0)
        ax.bar(x + offset, vals, width, color=color, alpha=0.85, label=ptype)
    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in COND_ORDER], fontsize=7)
    ax.set_ylabel("Acc@1")
    ax.set_title(f"Train vs paraphrase by condition ({lens})")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "final_vs_paraphrase_by_condition", lens=lens)


### Graph 4 — paraphrase layers after LoRA, tuned lens (pooled)

**Presents:** layer-wise % top-1 on paraphrase prompts, restricted to facts
that were correct on train both before and after LoRA, wrong on at least one
paraphrase before LoRA, and correct on the paraphrase after LoRA — i.e.
facts where LoRA specifically fixed the generalization gap.

**Hypothesis:** H1 on the "LoRA taught genuine generalization" subset — for
those facts, at what layer does the now-correct paraphrase answer emerge?

**Population:** train+paraphrase prompts, tuned lens, final checkpoint,
facts matching the filter above (pooled across conditions; typically a small
n).


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if traj is None or not path.exists():
    print("[notebook] missing data — skipping.")
else:
    t = traj[(traj["lens"] == lens) & (traj["variant"].isin(["base", "final"]))]
    train = t[t["prompt_type"] == "train"]
    para = t[t["prompt_type"] == "paraphrase"]
    if train.empty or para.empty:
        print("[notebook] no paraphrase rows — skipping.")
    else:
        tr = train.set_index(["fact_id", "variant"])
        pr = para.set_index(["fact_id", "variant"])
        base_train_ok = set(tr.loc[(slice(None), "base"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))
        final_train_ok = set(tr.loc[(slice(None), "final"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))
        base_para = pr.loc[(slice(None), "base"), :]
        base_para_bad = set(base_para.loc[
            ~base_para["traj_class"].isin(["late_only", "persistent"])].index.get_level_values(0))
        final_para_ok = set(pr.loc[(slice(None), "final"), :].query(
            "traj_class in ['late_only','persistent']").index.get_level_values(0))

        facts = base_train_ok & base_para_bad & final_para_ok & final_train_ok
        if not facts:
            print("[notebook] filter matched 0 facts — skipping.")
        else:
            para_final = para[(para["variant"] == "final") & (para["fact_id"].isin(facts))]
            idxs = set(para_final["prompt_idx"])
            df = pd.read_parquet(path)
            curve = viz._layer_top1_pooled(df, "final", lens, prompt_type="paraphrase", prompt_idxs=idxs)

            fig, ax = plt.subplots(figsize=(3.5, 2.8))
            ax.plot(curve["layer"], curve["pct_top1"], color="#0072B2",
                    marker="o", markevery=MARKER_EVERY, markersize=3)
            ax.set_xlabel("Transformer layer")
            ax.set_ylabel("% top-1 (paraphrase, post-LoRA)")
            ax.set_title(f"Paraphrase layers after LoRA (n_facts={len(facts)}, {lens})")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "paraphrase_layers_after_lora", lens=lens)


### Graph 5a — paraphrase layer delay, tuned lens (pooled)

**Presents:** histogram of `Δ = L_para − L_orig` (earliest layer on the
paraphrase minus on the original train prompt, post-LoRA); facts never
correct on either side are counted separately and excluded from the
histogram.

**Hypothesis:** H1's generalization-depth counterpart — even when LoRA
generalizes to a paraphrase, does it take extra layers to get there?

**Population:** final checkpoint, tuned lens, facts with a first_layer on
both train and (earliest) paraphrase, pooled across conditions.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["variant"] == "final")]
    train = sub[sub["prompt_type"] == "train"][["fact_id", "condition", "first_layer", "traj_class"]].rename(
        columns={"first_layer": "L_orig", "traj_class": "cls_orig"})
    para = (sub[sub["prompt_type"] == "paraphrase"]
            .groupby(["fact_id", "condition"], as_index=False)
            .agg(L_para=("first_layer", "min"),
                 cls_para=("traj_class", lambda s: "never" if (s == "never").all() else "ok")))
    joined = train.merge(para, on=["fact_id", "condition"], how="inner")
    never = joined[(joined["L_orig"].isna()) | (joined["L_para"].isna())]
    both = joined[joined["L_orig"].notna() & joined["L_para"].notna()].copy()
    both["delta"] = both["L_para"] - both["L_orig"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    if not both.empty:
        ax.hist(both["delta"], bins=20, color="#0072B2", alpha=0.8, edgecolor="white")
        ax.axvline(both["delta"].mean(), color="#D55E00", linestyle="--", label=f"mean={both['delta'].mean():.1f}")
    ax.set_xlabel(r"$\Delta = L_{para} - L_{orig}$")
    ax.set_ylabel("Facts")
    ax.set_title(f"Paraphrase layer delay ({lens})\nn={len(both)}, never-correct={len(never)}")
    if not both.empty:
        ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "paraphrase_layer_delay", lens=lens)


### Graph 5b — paraphrase layer delay by condition, tuned lens

**Presents:** boxplots of the same `Δ = L_para − L_orig`, split by
condition.

**Hypothesis:** H2's version of Graph 5a — does the generalization-depth
penalty differ between known/latent/unknown facts and synthetic ones?

**Population:** final checkpoint, tuned lens, same facts as Graph 5a, split
per condition.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["variant"] == "final")]
    train = sub[sub["prompt_type"] == "train"][["fact_id", "condition", "first_layer", "traj_class"]].rename(
        columns={"first_layer": "L_orig", "traj_class": "cls_orig"})
    para = (sub[sub["prompt_type"] == "paraphrase"]
            .groupby(["fact_id", "condition"], as_index=False)
            .agg(L_para=("first_layer", "min"),
                 cls_para=("traj_class", lambda s: "never" if (s == "never").all() else "ok")))
    joined = train.merge(para, on=["fact_id", "condition"], how="inner")
    both = joined[joined["L_orig"].notna() & joined["L_para"].notna()].copy()
    both["delta"] = both["L_para"] - both["L_orig"]

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    data, labels = [], []
    for cond in COND_ORDER:
        vals = both.loc[both["condition"] == cond, "delta"]
        if len(vals):
            data.append(vals)
            labels.append(CONDITION_LABELS[cond])
    if data:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True)
        for patch, cond in zip(bp["boxes"], [c for c in COND_ORDER if len(both[both["condition"] == c])]):
            patch.set_facecolor(COLORS[cond])
            patch.set_alpha(0.75)
    ax.set_ylabel(r"$\Delta = L_{para} - L_{orig}$")
    ax.set_title(f"Paraphrase delay by condition ({lens})")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "paraphrase_layer_delay_by_condition", lens=lens)


### Graph 6a — fact-state transitions, tuned lens (pooled Sankey)

**Presents:** an alluvial diagram of facts moving between three collapsed
states — `never` / `transient` / `survives` (late_only ∪ persistent) — from
base to final.

**Hypothesis:** H1 as a state-transition story: mass flowing from
`never`/`transient` into `survives` is the qualitative signature of LoRA
making facts stick.

**Population:** train prompts, tuned lens, all conditions pooled.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])

    def _flows(frame):
        wide = frame.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        if wide.empty or not {"base", "final"}.issubset(wide.columns):
            return {}
        counts = wide.groupby(["base", "final"]).size()
        return {(a, b): int(n) for (a, b), n in counts.items()}

    fig, ax = plt.subplots(figsize=(4.5, 3.0))
    viz._draw_sankey(ax, _flows(sub))
    ax.set_title(f"Fact-state transitions ({lens}, pooled)")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "fact_state_sankey", lens=lens)


### Graph 6b — fact-state transitions by condition, tuned lens

**Presents:** one Sankey panel per condition (same never/transient/survives
states).

**Hypothesis:** H2's version of Graph 6a — do known/latent/unknown/synthetic
facts show different transition patterns?

**Population:** train prompts, tuned lens, split per condition.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])

    def _flows(frame):
        wide = frame.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        if wide.empty or not {"base", "final"}.issubset(wide.columns):
            return {}
        counts = wide.groupby(["base", "final"]).size()
        return {(a, b): int(n) for (a, b), n in counts.items()}

    present = [c for c in COND_ORDER if c in sub["condition"].values]
    if not present:
        print("[notebook] no conditions present — skipping.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(3.2 * len(present), 3.0))
        if len(present) == 1:
            axes = [axes]
        for ax, cond in zip(axes, present):
            viz._draw_sankey(ax, _flows(sub[sub["condition"] == cond]))
            ax.set_title(CONDITION_LABELS[cond])
        fig.suptitle(f"Fact-state transitions by condition ({lens})", fontsize=10, y=1.02)
        fig.tight_layout(pad=0.4)
        show_and_save(fig, "fact_state_sankey_by_condition", lens=lens)


### Graph 7 — accuracy vs LoRA rank, tuned lens

**Presents:** Acc@1 vs LoRA rank $r$ (rank 0 = base), one line per condition,
from the rank-ablation sweep.

**Hypothesis:** H2's capacity angle — does adapter rank matter differently
for known vs synthetic facts (e.g. does synthetic need higher rank to reach
the same accuracy)?

**Population:** train prompts, tuned lens, `rank_ablation_summary.csv`
(re-trained adapters at each swept rank).


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "rank_ablation_summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cond in COND_ORDER:
        sub = df[df["condition"] == cond].sort_values("rank")
        if sub.empty:
            continue
        ax.plot(sub["rank"], sub["final_accuracy"], color=COLORS[cond], linestyle=LINE_STYLES[cond],
                marker=MARKERS[cond], markersize=4, label=CONDITION_LABELS[cond])
    ax.set_xlabel("LoRA rank $r$ (0 = base)")
    ax.set_ylabel("Acc@1")
    ax.set_title(f"Accuracy vs rank ({lens})")
    ax.set_ylim(-0.03, 1.03)
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "rank_final_accuracy", lens=lens)


### Graph 8 — layer-wise accuracy by rank, faceted by condition, tuned lens

**Presents:** one subplot per condition; within each, a layer-wise % top-1
curve per swept rank (viridis colormap, low→high rank).

**Hypothesis:** H1+H2 combined — at which layer does higher rank start to
help, and does that layer/threshold shift by condition?

**Population:** train prompts, tuned lens, `rank_ablation_layerwise.parquet`.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "rank_ablation_layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train")]
    present = [c for c in COND_ORDER if c in df["condition"].values]
    if not present:
        print("[notebook] no conditions present — skipping.")
    else:
        fig, axes = plt.subplots(1, len(present), figsize=(3.0 * len(present), 2.8), sharey=True)
        if len(present) == 1:
            axes = [axes]
        ranks = sorted(df["rank"].unique()) if "rank" in df.columns else sorted(df["variant"].unique())
        cmap = plt.cm.viridis(np.linspace(0.2, 0.9, len(ranks)))

        for ax, cond in zip(axes, present):
            sub = df[df["condition"] == cond]
            for color, rank in zip(cmap, ranks):
                rsub = sub[sub["rank"] == rank] if "rank" in sub.columns else sub[sub["variant"] == rank]
                curve = (rsub.groupby("layer")["in_top_1"].mean() * 100).reset_index()
                if curve.empty:
                    continue
                ax.plot(curve["layer"], curve["in_top_1"], color=color, label=str(rank), linewidth=1.2)
            ax.set_title(CONDITION_LABELS[cond])
            ax.set_xlabel("Layer")
        axes[0].set_ylabel("% top-1")
        axes[-1].legend(fontsize=6, title="rank", title_fontsize=6)
        fig.suptitle(f"Rank ablation layer accuracy ({lens})", fontsize=10, y=1.02)
        fig.tight_layout(pad=0.4)
        show_and_save(fig, "rank_layer_accuracy", lens=lens)


### Graph 9 — ΔAcc on detectable subset, tuned lens

**Presents:** `final − base` % top-1 per layer, restricted to the Graph
1 detectable subset (facts the base model could surface at some layer).

**Hypothesis:** H1 as a direct per-layer effect size on the cleanest
population — where exactly does LoRA add accuracy?

**Population:** train prompts, tuned lens, lens-detectable facts (same
filter as Graph 1).


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
traj = viz._load_traj(RESULTS_DIR)
if not path.exists() or traj is None:
    print("[notebook] missing data — skipping.")
else:
    idxs = viz.detectable_prompt_idxs(traj, lens)
    if not idxs:
        print("[notebook] no detectable prompts — skipping.")
    else:
        df = pd.read_parquet(path)
        base = viz._layer_top1_pooled(df, "base", lens, prompt_idxs=idxs)
        final = viz._layer_top1_pooled(df, "final", lens, prompt_idxs=idxs)
        merged = base.merge(final, on="layer", suffixes=("_base", "_final"))
        merged["delta"] = merged["pct_top1_final"] - merged["pct_top1_base"]

        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        ax.plot(merged["layer"], merged["delta"], color="#0072B2", marker="o", markevery=MARKER_EVERY, markersize=3)
        ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
        ax.set_xlabel("Transformer layer")
        ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
        ax.set_title(f"ΔAcc on detectable subset (n={len(idxs)}, {lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "delta_detectable", lens=lens)


### Graph 10a — ΔAcc by relation, tuned lens

**Presents:** bar chart of `final − base` Acc@1 at the last layer, one bar
per CounterFact relation, sorted descending.

**Hypothesis:** H2's relation-level breakdown — is the LoRA effect uniform
across relation types, or concentrated in a few?

**Population:** train prompts, tuned lens, last layer, joined against
`conditions.parquet` for the `relation` field.


In [ ]:
lens = "tuned"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    n_layers = int(df["layer"].max())
    final_layer = df[(df["lens"] == lens) & (df["prompt_type"] == "train") &
                      (df["layer"] == n_layers) & (df["variant"].isin(["base", "final"]))]
    joined = final_layer.merge(conds[["fact_id", "relation"]], on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] relation join empty — skipping.")
    else:
        acc = joined.groupby(["relation", "variant"])["in_top_1"].mean().unstack("variant")
        if not {"base", "final"}.issubset(acc.columns):
            print("[notebook] missing base/final for relations — skipping.")
        else:
            acc["delta"] = acc["final"] - acc["base"]
            acc = acc.sort_values("delta", ascending=False)

            fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(acc)), 3.0))
            ax.bar(range(len(acc)), acc["delta"] * 100, color="#0072B2", alpha=0.85)
            ax.set_xticks(range(len(acc)))
            ax.set_xticklabels(acc.index, rotation=90, fontsize=6)
            ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
            ax.set_title(f"ΔAcc by relation ({lens})")
            ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "delta_by_relation", lens=lens)


### Graph 10b — ΔAcc heatmap by relation × layer, tuned lens

**Presents:** heatmap of `final − base` % top-1, relations on the y-axis,
layers on the x-axis, diverging red/blue colormap.

**Hypothesis:** H2's fine-grained view of Graph 10a — for relations with a
big net effect, at which layer does that effect actually appear?

**Population:** train prompts, tuned lens, every layer, joined against
`conditions.parquet` for `relation`.


In [ ]:
lens = "tuned"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    lw = df[(df["lens"] == lens) & (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    lw = lw.merge(conds[["fact_id", "relation"]], on="fact_id", how="inner")
    if lw.empty:
        print("[notebook] relation join empty — skipping.")
    else:
        pivot = lw.groupby(["relation", "variant", "layer"])["in_top_1"].mean().unstack("variant")
        if not {"base", "final"}.issubset(pivot.columns):
            print("[notebook] missing base/final for relations — skipping.")
        else:
            pivot["delta"] = (pivot["final"] - pivot["base"]) * 100
            heat = pivot["delta"].unstack("layer")
            fig, ax = plt.subplots(figsize=(6.5, max(2.5, 0.22 * len(heat))))
            im = ax.imshow(heat.values, aspect="auto", cmap="RdBu_r",
                            vmin=-np.nanmax(np.abs(heat.values)), vmax=np.nanmax(np.abs(heat.values)))
            ax.set_yticks(range(len(heat)))
            ax.set_yticklabels(heat.index, fontsize=6)
            ax.set_xlabel("Layer")
            ax.set_title(f"ΔAcc heatmap by relation × layer ({lens})")
            fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Δ pp")
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "delta_by_relation_heatmap", lens=lens)


## Synthetic facts S1–S9 — existing vs fabricated pseudo-entity facts

`existing` = known ∪ latent ∪ unknown (real CounterFact facts); `synthetic` =
fabricated pseudo-entity facts built from the same relation templates, which
the base model cannot have seen anywhere. This is H2's control: any
"generalization" the model shows on synthetic facts must come from LoRA
training, not prior knowledge.


### S1 — layer top-1, existing vs synthetic, logit lens

**Presents:** % top-1 per layer, base (dashed) vs LoRA (solid), one color for
existing facts and one for synthetic facts.

**Hypothesis:** H2 control — synthetic facts should show *no* base-model
signal (flat/low base curve) and the whole LoRA effect should be learned
from scratch during fine-tuning.

**Population:** train prompts, logit lens, existing vs synthetic groups.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
        for variant, ls in (("base", "--"), ("final", "-")):
            sub = df[(df["group"] == group) & (df["variant"] == variant) &
                     (df["lens"] == lens) & (df["prompt_type"] == "train")]
            if sub.empty:
                continue
            curve = sub.groupby("layer")["in_top_1"].mean() * 100
            ax.plot(curve.index, curve.values, color=color, linestyle=ls,
                    label=f"{group}/{variant}", linewidth=1.5)
    ax.set_xlabel("Layer")
    ax.set_ylabel("% top-1")
    ax.set_title(f"S1: existing vs synthetic ({lens})")
    ax.legend(fontsize=6)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S2 — Δ log-prob, existing vs synthetic, logit lens

**Presents:** mean `answer_logprob(final) − answer_logprob(base)` per layer,
one curve for existing facts and one for synthetic facts.

**Hypothesis:** H1/H2 control — does LoRA add log-probability at the same
layers for facts it's memorizing from scratch (synthetic) as for facts it
partially already knew (existing)?

**Population:** train prompts, logit lens, existing vs synthetic groups.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
        sub = df[df["group"] == group]
        mean_lp = sub.groupby(["variant", "layer"])["answer_logprob"].mean().unstack(0)
        if not {"base", "final"}.issubset(mean_lp.columns):
            continue
        ax.plot(mean_lp.index, mean_lp["final"] - mean_lp["base"], color=color, label=group,
                marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Δ mean log-prob")
    ax.set_title(f"S2: Δ log-prob existing vs synthetic ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_delta_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S3 — first-layer distributions, existing vs synthetic, logit lens

**Presents:** boxplots of `first_layer` (final variant), existing vs
synthetic, annotated with the fraction of facts that are `never` correct in
each group.

**Hypothesis:** H2 control — once LoRA has actually learned a synthetic
fact, does it settle at a similar depth to real facts, or does fabricated
knowledge live deeper/shallower?

**Population:** train prompts, logit lens, final variant, existing vs
synthetic groups.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") & (traj["variant"] == "final")].copy()
    sub["group"] = viz.existing_vs_synthetic(sub["condition"])
    never_frac = sub.groupby("group")["traj_class"].apply(lambda s: (s == "never").mean())

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    data, labels = [], []
    for group in ("existing", "synthetic"):
        vals = sub.loc[sub["group"] == group, "first_layer"].dropna()
        if len(vals):
            data.append(vals)
            nf = float(never_frac.get(group, 0))
            labels.append(f"{group}\n(never={nf:.0%})")
    if data:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True)
        for patch, group in zip(bp["boxes"], [g for g in ("existing", "synthetic")
                                               if len(sub.loc[sub["group"] == g, "first_layer"].dropna())]):
            patch.set_facecolor(COLORS[group])
            patch.set_alpha(0.75)
    ax.set_ylabel("first_layer (final)")
    ax.set_title(f"S3: first-layer existing vs synthetic ({lens})")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "first_layer_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S4 — accuracy over checkpoints, existing vs synthetic, logit lens

**Presents:** Acc@1 vs training step, one line for existing facts and one
for synthetic facts. Skipped if the run has no intermediate checkpoints.

**Hypothesis:** H2's learning-dynamics control — synthetic facts (learned
purely from the LoRA data) may take longer to reach the same accuracy as
existing facts (which may already have partial prior support).

**Population:** train prompts, logit lens, every checkpoint through final,
existing vs synthetic groups.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    steps = s[s["variant"].str.startswith("step_", na=False)]
    if steps.empty:
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]
        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
            g = use[use["group"] == group].groupby("step")["final_accuracy"].mean().reset_index()
            ax.plot(g["step"], g["final_accuracy"], color=color, marker="o", markersize=4, label=group)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Acc@1")
        ax.set_title(f"S4: Acc over checkpoints ({lens})")
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "acc_over_checkpoints", prefix="new_version_synthetic", lens=lens)


### S5 — first-layer over checkpoints, existing vs synthetic, logit lens

**Presents:** mean `first_layer` vs training step, one line per group,
annotated with the final-step `never`-correct fraction.

**Hypothesis:** H1's learning-dynamics control — does the answer's layer
depth for synthetic facts converge toward existing facts' depth as training
progresses, or stay persistently different?

**Population:** train prompts, logit lens, every checkpoint through final,
existing vs synthetic groups.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    if not s["variant"].str.startswith("step_", na=False).any():
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]
        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
            g = (use[use["group"] == group].groupby("step")
                 .agg(mean_first=("mean_first_layer", "mean"), frac_never=("frac_never_top1", "mean"))
                 .reset_index())
            ax.plot(g["step"], g["mean_first"], color=color, marker="o", markersize=4, label=group)
            if not g.empty:
                last = g.iloc[-1]
                ax.annotate(f"never={last['frac_never']:.0%}", (last["step"], last["mean_first"]),
                            textcoords="offset points", xytext=(4, 4), fontsize=6, color=color)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Mean first_layer")
        ax.set_title(f"S5: first-layer over checkpoints ({lens})")
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "first_layer_over_checkpoints", prefix="new_version_synthetic", lens=lens)


### S6 — paraphrase accuracy, existing vs synthetic, logit lens

**Presents:** two bars — mean paraphrase Acc@1 for existing vs synthetic
facts at the final checkpoint.

**Hypothesis:** H2's strongest generalization control — synthetic facts have
zero prior support, so any paraphrase accuracy there is unambiguous evidence
LoRA taught a *reusable* association rather than a surface-level pattern.

**Population:** held-out paraphrase prompts, logit lens, final checkpoint,
existing vs synthetic groups.


In [ ]:
lens = "logit"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final") & (s["prompt_type"] == "paraphrase")].copy()
    if s.empty:
        print("[notebook] no paraphrase rows — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        g = s.groupby("group")["final_accuracy"].mean()
        fig, ax = plt.subplots(figsize=(3.0, 2.8))
        groups = [x for x in ("existing", "synthetic") if x in g.index]
        ax.bar(range(len(groups)), [g[x] for x in groups], color=[COLORS[x] for x in groups], alpha=0.85)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels(groups)
        ax.set_ylabel("Paraphrase Acc@1")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"S6: paraphrase Acc ({lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "paraphrase_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S7 — state transitions, existing vs synthetic, logit lens

**Presents:** two Sankey panels (never/transient/survives), one for existing
facts and one for synthetic facts.

**Hypothesis:** H2 control, state-transition view of S1 — does the
never→survives flow look similar for facts learned from scratch (synthetic)
vs facts with partial prior support (existing)?

**Population:** train prompts, logit lens, existing vs synthetic groups.


In [ ]:
lens = "logit"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])
    sub["group"] = viz.existing_vs_synthetic(sub["condition"])

    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
    for ax, group in zip(axes, ("existing", "synthetic")):
        g = sub[sub["group"] == group]
        wide = g.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        flows = {}
        if not wide.empty and {"base", "final"}.issubset(wide.columns):
            counts = wide.groupby(["base", "final"]).size()
            flows = {(a, b): int(n) for (a, b), n in counts.items()}
        viz._draw_sankey(ax, flows)
        ax.set_title(group)
    fig.suptitle(f"S7: state transitions ({lens})", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4)
    show_and_save(fig, "sankey_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S9a — synthetic accuracy by relation, logit lens

**Presents:** bar chart of final-layer Acc@1 by relation, synthetic facts
only, sorted descending.

**Hypothesis:** H2's relation-level breakdown restricted to synthetic
facts — are some relation templates easier for LoRA to learn from scratch
than others?

**Population:** train prompts, logit lens, synthetic condition only, joined
against `conditions.parquet` for `relation`.


In [ ]:
lens = "logit"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    syn_conds = conds[conds["condition"] == "synthetic"][["fact_id", "relation"]]
    df = pd.read_parquet(path)
    df = df[(df["condition"] == "synthetic") & (df["lens"] == lens) &
            (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    joined = df.merge(syn_conds, on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] synthetic×relation join empty — skipping.")
    else:
        n_layers = int(joined["layer"].max())
        final_acc = (joined[(joined["variant"] == "final") & (joined["layer"] == n_layers)]
                     .groupby("relation")["in_top_1"].mean().sort_values(ascending=False))

        fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(final_acc)), 3.0))
        ax.bar(range(len(final_acc)), final_acc.values * 100, color=COLORS["synthetic"], alpha=0.85)
        ax.set_xticks(range(len(final_acc)))
        ax.set_xticklabels(final_acc.index, rotation=90, fontsize=6)
        ax.set_ylabel("Acc@1 (%)")
        ax.set_title(f"S9: synthetic Acc by relation ({lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_by_relation", prefix="new_version_synthetic", lens=lens)


### S9b — synthetic ΔAcc by relation, logit lens

**Presents:** bar chart of `final − base` Acc@1 by relation, synthetic facts
only.

**Hypothesis:** same as S9a but isolating the LoRA-induced change rather
than the raw final accuracy.

**Population:** train prompts, logit lens, synthetic condition only, last
layer, joined against `conditions.parquet`.


In [ ]:
lens = "logit"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    syn_conds = conds[conds["condition"] == "synthetic"][["fact_id", "relation"]]
    df = pd.read_parquet(path)
    df = df[(df["condition"] == "synthetic") & (df["lens"] == lens) &
            (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    joined = df.merge(syn_conds, on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] synthetic×relation join empty — skipping.")
    else:
        n_layers = int(joined["layer"].max())
        acc = (joined[joined["layer"] == n_layers].groupby(["relation", "variant"])["in_top_1"]
               .mean().unstack("variant"))
        if not {"base", "final"}.issubset(acc.columns):
            print("[notebook] missing base/final — skipping.")
        else:
            acc["delta"] = (acc["final"] - acc["base"]) * 100
            acc = acc.sort_values("delta", ascending=False)
            fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(acc)), 3.0))
            ax.bar(range(len(acc)), acc["delta"], color=COLORS["synthetic"], alpha=0.85)
            ax.set_xticks(range(len(acc)))
            ax.set_xticklabels(acc.index, rotation=90, fontsize=6)
            ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
            ax.set_title(f"S9: synthetic ΔAcc by relation ({lens})")
            ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "layer_by_relation_delta", prefix="new_version_synthetic", lens=lens)


### S1 — layer top-1, existing vs synthetic, tuned lens

**Presents:** % top-1 per layer, base (dashed) vs LoRA (solid), one color for
existing facts and one for synthetic facts.

**Hypothesis:** H2 control — synthetic facts should show *no* base-model
signal (flat/low base curve) and the whole LoRA effect should be learned
from scratch during fine-tuning.

**Population:** train prompts, tuned lens, existing vs synthetic groups.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
        for variant, ls in (("base", "--"), ("final", "-")):
            sub = df[(df["group"] == group) & (df["variant"] == variant) &
                     (df["lens"] == lens) & (df["prompt_type"] == "train")]
            if sub.empty:
                continue
            curve = sub.groupby("layer")["in_top_1"].mean() * 100
            ax.plot(curve.index, curve.values, color=color, linestyle=ls,
                    label=f"{group}/{variant}", linewidth=1.5)
    ax.set_xlabel("Layer")
    ax.set_ylabel("% top-1")
    ax.set_title(f"S1: existing vs synthetic ({lens})")
    ax.legend(fontsize=6)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_top1_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S2 — Δ log-prob, existing vs synthetic, tuned lens

**Presents:** mean `answer_logprob(final) − answer_logprob(base)` per layer,
one curve for existing facts and one for synthetic facts.

**Hypothesis:** H1/H2 control — does LoRA add log-probability at the same
layers for facts it's memorizing from scratch (synthetic) as for facts it
partially already knew (existing)?

**Population:** train prompts, tuned lens, existing vs synthetic groups.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "layerwise.parquet"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_parquet(path)
    df["group"] = viz.existing_vs_synthetic(df["condition"])
    df = df[(df["lens"] == lens) & (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
        sub = df[df["group"] == group]
        mean_lp = sub.groupby(["variant", "layer"])["answer_logprob"].mean().unstack(0)
        if not {"base", "final"}.issubset(mean_lp.columns):
            continue
        ax.plot(mean_lp.index, mean_lp["final"] - mean_lp["base"], color=color, label=group,
                marker="o", markevery=MARKER_EVERY, markersize=3)
    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Δ mean log-prob")
    ax.set_title(f"S2: Δ log-prob existing vs synthetic ({lens})")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "layer_delta_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S3 — first-layer distributions, existing vs synthetic, tuned lens

**Presents:** boxplots of `first_layer` (final variant), existing vs
synthetic, annotated with the fraction of facts that are `never` correct in
each group.

**Hypothesis:** H2 control — once LoRA has actually learned a synthetic
fact, does it settle at a similar depth to real facts, or does fabricated
knowledge live deeper/shallower?

**Population:** train prompts, tuned lens, final variant, existing vs
synthetic groups.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") & (traj["variant"] == "final")].copy()
    sub["group"] = viz.existing_vs_synthetic(sub["condition"])
    never_frac = sub.groupby("group")["traj_class"].apply(lambda s: (s == "never").mean())

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    data, labels = [], []
    for group in ("existing", "synthetic"):
        vals = sub.loc[sub["group"] == group, "first_layer"].dropna()
        if len(vals):
            data.append(vals)
            nf = float(never_frac.get(group, 0))
            labels.append(f"{group}\n(never={nf:.0%})")
    if data:
        bp = ax.boxplot(data, tick_labels=labels, patch_artist=True)
        for patch, group in zip(bp["boxes"], [g for g in ("existing", "synthetic")
                                               if len(sub.loc[sub["group"] == g, "first_layer"].dropna())]):
            patch.set_facecolor(COLORS[group])
            patch.set_alpha(0.75)
    ax.set_ylabel("first_layer (final)")
    ax.set_title(f"S3: first-layer existing vs synthetic ({lens})")
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "first_layer_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S4 — accuracy over checkpoints, existing vs synthetic, tuned lens

**Presents:** Acc@1 vs training step, one line for existing facts and one
for synthetic facts. Skipped if the run has no intermediate checkpoints.

**Hypothesis:** H2's learning-dynamics control — synthetic facts (learned
purely from the LoRA data) may take longer to reach the same accuracy as
existing facts (which may already have partial prior support).

**Population:** train prompts, tuned lens, every checkpoint through final,
existing vs synthetic groups.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    steps = s[s["variant"].str.startswith("step_", na=False)]
    if steps.empty:
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]
        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
            g = use[use["group"] == group].groupby("step")["final_accuracy"].mean().reset_index()
            ax.plot(g["step"], g["final_accuracy"], color=color, marker="o", markersize=4, label=group)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Acc@1")
        ax.set_title(f"S4: Acc over checkpoints ({lens})")
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "acc_over_checkpoints", prefix="new_version_synthetic", lens=lens)


### S5 — first-layer over checkpoints, existing vs synthetic, tuned lens

**Presents:** mean `first_layer` vs training step, one line per group,
annotated with the final-step `never`-correct fraction.

**Hypothesis:** H1's learning-dynamics control — does the answer's layer
depth for synthetic facts converge toward existing facts' depth as training
progresses, or stay persistently different?

**Population:** train prompts, tuned lens, every checkpoint through final,
existing vs synthetic groups.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["prompt_type"] == "train")].copy()
    if not s["variant"].str.startswith("step_", na=False).any():
        print("[notebook] no intermediate checkpoints — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        use = s[s["variant"].str.startswith("step_", na=False) | (s["variant"] == "final")]
        fig, ax = plt.subplots(figsize=(3.5, 2.8))
        for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
            g = (use[use["group"] == group].groupby("step")
                 .agg(mean_first=("mean_first_layer", "mean"), frac_never=("frac_never_top1", "mean"))
                 .reset_index())
            ax.plot(g["step"], g["mean_first"], color=color, marker="o", markersize=4, label=group)
            if not g.empty:
                last = g.iloc[-1]
                ax.annotate(f"never={last['frac_never']:.0%}", (last["step"], last["mean_first"]),
                            textcoords="offset points", xytext=(4, 4), fontsize=6, color=color)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Mean first_layer")
        ax.set_title(f"S5: first-layer over checkpoints ({lens})")
        ax.legend(fontsize=7)
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "first_layer_over_checkpoints", prefix="new_version_synthetic", lens=lens)


### S6 — paraphrase accuracy, existing vs synthetic, tuned lens

**Presents:** two bars — mean paraphrase Acc@1 for existing vs synthetic
facts at the final checkpoint.

**Hypothesis:** H2's strongest generalization control — synthetic facts have
zero prior support, so any paraphrase accuracy there is unambiguous evidence
LoRA taught a *reusable* association rather than a surface-level pattern.

**Population:** held-out paraphrase prompts, tuned lens, final checkpoint,
existing vs synthetic groups.


In [ ]:
lens = "tuned"
path = RESULTS_DIR / "summary.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    s = pd.read_csv(path)
    s = s[(s["lens"] == lens) & (s["variant"] == "final") & (s["prompt_type"] == "paraphrase")].copy()
    if s.empty:
        print("[notebook] no paraphrase rows — skipping.")
    else:
        s["group"] = viz.existing_vs_synthetic(s["condition"])
        g = s.groupby("group")["final_accuracy"].mean()
        fig, ax = plt.subplots(figsize=(3.0, 2.8))
        groups = [x for x in ("existing", "synthetic") if x in g.index]
        ax.bar(range(len(groups)), [g[x] for x in groups], color=[COLORS[x] for x in groups], alpha=0.85)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels(groups)
        ax.set_ylabel("Paraphrase Acc@1")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"S6: paraphrase Acc ({lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "paraphrase_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S7 — state transitions, existing vs synthetic, tuned lens

**Presents:** two Sankey panels (never/transient/survives), one for existing
facts and one for synthetic facts.

**Hypothesis:** H2 control, state-transition view of S1 — does the
never→survives flow look similar for facts learned from scratch (synthetic)
vs facts with partial prior support (existing)?

**Population:** train prompts, tuned lens, existing vs synthetic groups.


In [ ]:
lens = "tuned"
traj = viz._load_traj(RESULTS_DIR)
if traj is None:
    print("[notebook] no trajectory — skipping.")
else:
    sub = traj[(traj["lens"] == lens) & (traj["prompt_type"] == "train") &
               (traj["variant"].isin(["base", "final"]))].copy()
    sub["state"] = viz.three_state(sub["traj_class"])
    sub["group"] = viz.existing_vs_synthetic(sub["condition"])

    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
    for ax, group in zip(axes, ("existing", "synthetic")):
        g = sub[sub["group"] == group]
        wide = g.pivot_table(index="prompt_idx", columns="variant", values="state", aggfunc="first").dropna()
        flows = {}
        if not wide.empty and {"base", "final"}.issubset(wide.columns):
            counts = wide.groupby(["base", "final"]).size()
            flows = {(a, b): int(n) for (a, b), n in counts.items()}
        viz._draw_sankey(ax, flows)
        ax.set_title(group)
    fig.suptitle(f"S7: state transitions ({lens})", fontsize=10, y=1.02)
    fig.tight_layout(pad=0.4)
    show_and_save(fig, "sankey_existing_vs_synthetic", prefix="new_version_synthetic", lens=lens)


### S9a — synthetic accuracy by relation, tuned lens

**Presents:** bar chart of final-layer Acc@1 by relation, synthetic facts
only, sorted descending.

**Hypothesis:** H2's relation-level breakdown restricted to synthetic
facts — are some relation templates easier for LoRA to learn from scratch
than others?

**Population:** train prompts, tuned lens, synthetic condition only, joined
against `conditions.parquet` for `relation`.


In [ ]:
lens = "tuned"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    syn_conds = conds[conds["condition"] == "synthetic"][["fact_id", "relation"]]
    df = pd.read_parquet(path)
    df = df[(df["condition"] == "synthetic") & (df["lens"] == lens) &
            (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    joined = df.merge(syn_conds, on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] synthetic×relation join empty — skipping.")
    else:
        n_layers = int(joined["layer"].max())
        final_acc = (joined[(joined["variant"] == "final") & (joined["layer"] == n_layers)]
                     .groupby("relation")["in_top_1"].mean().sort_values(ascending=False))

        fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(final_acc)), 3.0))
        ax.bar(range(len(final_acc)), final_acc.values * 100, color=COLORS["synthetic"], alpha=0.85)
        ax.set_xticks(range(len(final_acc)))
        ax.set_xticklabels(final_acc.index, rotation=90, fontsize=6)
        ax.set_ylabel("Acc@1 (%)")
        ax.set_title(f"S9: synthetic Acc by relation ({lens})")
        fig.tight_layout(pad=0.5)
        show_and_save(fig, "layer_by_relation", prefix="new_version_synthetic", lens=lens)


### S9b — synthetic ΔAcc by relation, tuned lens

**Presents:** bar chart of `final − base` Acc@1 by relation, synthetic facts
only.

**Hypothesis:** same as S9a but isolating the LoRA-induced change rather
than the raw final accuracy.

**Population:** train prompts, tuned lens, synthetic condition only, last
layer, joined against `conditions.parquet`.


In [ ]:
lens = "tuned"
conds = viz._load_conditions(RESULTS_DIR, OUTPUT_DIR)
path = RESULTS_DIR / "layerwise.parquet"
if conds is None:
    print("[notebook] conditions.parquet not found for this run — skipping.")
elif not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    syn_conds = conds[conds["condition"] == "synthetic"][["fact_id", "relation"]]
    df = pd.read_parquet(path)
    df = df[(df["condition"] == "synthetic") & (df["lens"] == lens) &
            (df["prompt_type"] == "train") & (df["variant"].isin(["base", "final"]))]
    joined = df.merge(syn_conds, on="fact_id", how="inner")
    if joined.empty:
        print("[notebook] synthetic×relation join empty — skipping.")
    else:
        n_layers = int(joined["layer"].max())
        acc = (joined[joined["layer"] == n_layers].groupby(["relation", "variant"])["in_top_1"]
               .mean().unstack("variant"))
        if not {"base", "final"}.issubset(acc.columns):
            print("[notebook] missing base/final — skipping.")
        else:
            acc["delta"] = (acc["final"] - acc["base"]) * 100
            acc = acc.sort_values("delta", ascending=False)
            fig, ax = plt.subplots(figsize=(max(4.0, 0.25 * len(acc)), 3.0))
            ax.bar(range(len(acc)), acc["delta"], color=COLORS["synthetic"], alpha=0.85)
            ax.set_xticks(range(len(acc)))
            ax.set_xticklabels(acc.index, rotation=90, fontsize=6)
            ax.set_ylabel(r"$\Delta$ Acc@1 (pp)")
            ax.set_title(f"S9: synthetic ΔAcc by relation ({lens})")
            ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
            fig.tight_layout(pad=0.5)
            show_and_save(fig, "layer_by_relation_delta", prefix="new_version_synthetic", lens=lens)


### S8 — patching flip rate, existing vs synthetic (lens-free)

**Presents:** flip rate vs layer under activation patching, one curve for
existing facts and one for synthetic facts.

**Hypothesis:** H2's causal control — lens-free confirmation of S1: does the
causal locus of the LoRA update sit at a similar depth for synthetic facts
as for real ones?

**Population:** final LoRA checkpoint, all layers, existing vs synthetic
groups, lens-free.


In [ ]:
path = RESULTS_DIR / "patching.csv"
if not path.exists():
    print(f"[notebook] {path} not found — skipping.")
else:
    df = pd.read_csv(path)
    df = df[(df["variant"] == "final") & (df["layer"] >= 0)].copy()
    df["group"] = viz.existing_vs_synthetic(df["condition"])

    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for group, color in (("existing", COLORS["existing"]), ("synthetic", COLORS["synthetic"])):
        sub = df[df["group"] == group]
        rate = sub.groupby("layer")["flipped"].mean()
        ax.plot(rate.index, rate.values, color=color, marker="o", markevery=MARKER_EVERY,
                markersize=3, label=group)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Flip rate")
    ax.set_title("S8: patching flip rate existing vs synthetic")
    ax.legend(fontsize=7)
    fig.tight_layout(pad=0.5)
    show_and_save(fig, "patching_existing_vs_synthetic", prefix="new_version_synthetic")
